# Demand model v0: what we can train now, and what data to ask for

Two questions:

1. Which of the four models (daily demand, night types, drink mix, reorder math) can we train with the data we already have?
2. What should we ask Kieran for? Each candidate data source is scored by how much it improves the daily demand forecast.

**Run it:** from `modeling/`, `uv sync` then `uv run jupyter lab`. Needs Kieran's exports in `data/sawtooth/` (gitignored) and `dartmouth_calendar_2025_2026.csv` (public dates, committed here).

## Answer

**Can we train the four models?** Model 1 (daily demand) and model 2 (night types), yes. Model 3 (drink mix) and model 4 (reorder math), no: the data they need doesn't exist yet. Section 7 lists what to ask for.

**How model 1 does.** WAPE pooled over the same three test periods as ChatGPT's scorecard. Every forecast from our models is made up to six months ahead, with no recent sales used.

| Model | WAPE, all 3 folds | Event days only |
|---|---|---|
| Weekday median (baseline; same as ChatGPT's) | 41.3% | 36.8% |
| ChatGPT's best multi-day model besides the baseline (Ridge) | 44.1% | |
| ChatGPT's best 1-day-ahead model (uses yesterday's sales) | 39.7% | |
| Ours, M2: weekday + 2026 level + Dartmouth calendar | 39.2% | 37.8% |
| Ours, two-stage event model, λ=10 (2b) | 38.9% | 36.2% |
| Ours, M2 + private-event dates (2b) | 38.7% | 36.0% |
| Ours, XGBoost alone with every input (2d) | 39.8% | 34.8% |
| **Ours, hybrid: linear baseline + XGBoost event effects (2d)** | **38.0%** | **34.0%** |
| Ours, M4: M2 + shows + big-ticket days (oracle upper bound) | 36.7% | |

On the 19 event days in 2026 that also happened in 2025, averaging the model with last year's same event cuts error from 23.7% to 19.8% (2c).

**Rolling backtest (section 8).** Refitting every 3 months after the first year and forecasting 7 or 28 days ahead: hybrid daily error about 35%, **weekly about 18-19%** (weekday median 23-24%). Refitting monthly instead changes results by about a point. The 80% ranges contained 73-77% of days, so they run slightly narrow.

**Holidays (section 8b, model v2).** Thanksgiving Eve, Dec 23 and New Year's Eve ran 3-4x a normal day in 2025; most other holidays show no effect, and five are effectively closures. v2 adds a big-night effect (+170%, shrinkage chosen by leave-one-out) and Halloween (+16%). It can't be backtested because those nights only occur in data no test period trains on.

**People, not dollars (section 8c, model v3).** Kieran scales staff, ordering and prep to how busy he'll be, so the target that matters is head count. Forecasting guests with the same model and inputs cuts the share of weeks more than 20% off from 32% to 8% and the typical-week error from 11.3% to 9.4% (a week ahead). It beats sales in every period of both backtests. This is the recommended model.

**Frozen forecasts (sections 9-10).** Oct 22 to Dec 31, 2026, made Nov 1 from data through Oct 21: v1 (sales, no holidays), v2 (sales, with holidays) and v3 (guests, with holidays), each saved in `forecasts/` and committed to git. Section 10 scores all three once Kieran's next export arrives.

**Booked private events (section 9d).** On nights Kieran has booked as a full buyout, the forecast shows "booked: private event" instead of a walk-in number, because buyout guests are billed by invoice and never show up as walk-ins. Until his calendar is available, a rule built from Toast's own records stands in (normally closed days, reunions, weekend formals). It catches only 6 of the 14 past nights where walk-ins collapsed, so his calendar matters.

**The product (sections 9e and 9g).** A week-by-week calendar of expected guests for Oct 22 to Dec 31, 2026 (`outputs/sawtooth_calendar_2026-10-22_to_2026-12-31.html`). Each week gets a total with a range that held up on held-out weeks (75%); each night gets its expected count, how busy that is against a normal night this fall, and the reasons (holidays, Dartmouth events, home games, bookings). Nightly ranges were dropped: ranges that held up would span about half to 1.5 times the forecast.

**Model v5 (section 9f).** Since lunch service ended (assumed July 1, 2026), v3 runs high on Wednesdays and Thursdays and low on Saturdays. v5 gives each weekday its own level after that date. In a check on Sep 1-Oct 21 it cut that bias but added new bias on Tuesdays and Fridays and had higher weekly error (8.1% vs 7.4%), so the calendar stays on v3. v5 is frozen for November-December, its real test.

**Log-scale baseline (section 11).** Fitting the baseline to log(guests), so weekday, term and the 2026 level act as percentages, didn't help: weekly error was 11.5% vs 11.3% a week ahead and 12.3% vs 12.0% four weeks ahead. It failed a rule set before its scores were shown, so the next forecast run keeps the raw baseline.

**Caveats.** The gain over the baseline is concentrated in one period (2026 H2: 43.6% to about 30-32%); in 2026 H1 the calendar didn't help. The 80% forecast ranges held in 2025 H2 (82%) but covered only 71% in 2026 H1, because sales fell about 22% year over year, which a model trained on 2025 can't anticipate. M4 flags days using their own outcome (5 of its 28 days look like payments, the rest like private parties), so it's an upper bound, not a usable model. The features, λ, and the choice among three XGBoost designs were all made by people who had already looked at these test periods, so some optimism applies. The honest next test is prospective: forecast November now and score it when the data arrives.

**The model forecasts total restaurant sales, not drinks.** That's what the data supports today. The project's goal is drink reorder points, which needs the asks in section 7.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import QuantileRegressor

warnings.filterwarnings("ignore", message="Workbook contains no default style")

DATA = Path("../data/sawtooth")
CALENDAR_CSV = Path("dartmouth_calendar_2025_2026.csv")
assert DATA.exists(), "Put Kieran's exports in data/sawtooth/ (see data/README.md)"

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)

# dataviz reference palette (light); blue/orange pair validated for CVD
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, BLUE_LIGHT = "#2a78d6", "#eb6834", "#cde2fb"
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": AXIS,
    "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False, "figure.dpi": 110,
})

## 1. Inputs

- **Daily sales** from the `Sales by day` sheets (see `eda/sawtooth_chatgpt_audit.ipynb` for the audit).
- **Dartmouth calendar** (public): term dates, finals, Commencement, Green Key, Homecoming, Winter Carnival, Family Weekend, orientation, home football.
- **Sawtooth show dates** and **private-event dates**, parsed from item names in Product Mix: tickets ("5/17 9PM CANOPY") and Event-category payments ("6/13 Grad Party Deposit"). Names usually have no year, so each date gets the year that puts it closest to the middle of its export, which handles presales and deposits. Shows without tickets (DJ or dance nights) are missing. Private events are booked ahead in real life, so their dates are a fair forecast input; ours is a partial list rebuilt from payment names.
- **Big-ticket days (oracle)**: days with net sales per order of $150 or more (the median day is about $42). This uses the day's own outcome, so it's **not** a usable forecast input. Only 5 of the 28 look like payment-only days (1-5 orders); the rest are nights with unusually high spend per order, most likely private parties. Treat M4 as an upper bound on what knowing every private party in advance could buy.

In [ ]:
def load_daily():
    frames = []
    # Oldest period first: the export whose period ends latest was pulled most recently, so its rows win below.
    for f in sorted(DATA.glob("SalesSummary_*.xlsx"), key=lambda p: (p.stem.split("_")[2], p.stem.split("_")[1])):
        frames.append(pd.read_excel(f, sheet_name="Sales by day"))
    d = pd.concat(frames, ignore_index=True)
    d["date"] = pd.to_datetime(d["yyyyMMdd"].astype(str), format="%Y%m%d")
    d = d.drop(columns="yyyyMMdd").rename(
        columns={"Net sales": "net_sales", "Total orders": "orders", "Total guests": "guests"})
    repeated = d[d.duplicated("date", keep=False)]
    if len(repeated):
        # Exports can overlap. A date in more than one export keeps the row from the newest export (the one that ends latest).
        differ = repeated.groupby("date")[["net_sales", "orders", "guests"]].nunique(dropna=False).gt(1).any(axis=1).sum()
        print(f"{repeated.date.nunique()} dates appear in more than one Sales Summary export ({differ} differ in net sales, "
              f"orders or guests); keeping the newest export's row for each.")
        d = d.drop_duplicates("date", keep="last").sort_values("date", ignore_index=True)
    return d


daily = load_daily()
FORECAST_END = pd.Timestamp("2026-12-31")   # the calendar runs past the data so forecast days get the same features
cal = pd.DataFrame({"date": pd.date_range(daily.date.min(), max(daily.date.max(), FORECAST_END))}).merge(
    daily, on="date", how="left")
cal["weekday"] = cal.date.dt.day_name()
print(f"Sales data {daily.date.min():%Y-%m-%d} to {daily.date.max():%Y-%m-%d}: {int(cal.net_sales.notna().sum())} days "
      f"with numeric sales. Calendar extended to {cal.date.max():%Y-%m-%d} for the forecast.")

In [ ]:
def redact(name):
    # Item names can contain customer or group names, which must not appear in outputs. Keep only dates,
    # invoice numbers and generic booking words; replace each run of other words with a single "…".
    keep = r"[\d/\\#.]+[A-Z]?|invoice|deposit|payment|remainder|installment|party|grad|formal|reunion|catering|delivery|event"
    words = [w if re.fullmatch(keep, w, re.I) else "…" for w in str(name).split()]
    return " ".join(w for i, w in enumerate(words) if w != "…" or i == 0 or words[i - 1] != "…")


def dated_item_names(sheet, select):
    # Find M/D (or M/D/YY) dates inside item names in every 2025-26 Product Mix export. Names rarely
    # include a year, so use the year that puts the date closest to the middle of the export.
    rows = []
    for p in sorted(DATA.glob("ProductMix_202[56]-*.xlsx")):
        start, end = (pd.Timestamp(x) for x in p.stem.split("_")[1:3])
        mid = start + (end - start) / 2
        for name in select(pd.read_excel(p, sheet_name=sheet)).astype(str):
            m = re.search(r"\b(\d{1,2})[/\\](\d{1,2})(?:[/\\](\d{2,4}))?\b", name)
            if not m:
                continue
            years = [int(m[3]) + (2000 if len(m[3]) == 2 else 0)] if m[3] else [start.year - 1, start.year, start.year + 1]
            try:
                candidates = [pd.Timestamp(y, int(m[1]), int(m[2])) for y in years]
            except ValueError:   # not a real date, e.g. "2/30"
                continue
            rows.append({"date": min(candidates, key=lambda c: abs(c - mid)), "name": name})
    out = pd.DataFrame(rows)
    return out[out.date.between(cal.date.min(), cal.date.max())]


shows = dated_item_names("All levels", lambda al: al.loc[
    (al["Type"] == "menuItem") & (al["Menu"] == "TICKETS") & (al["Qty sold"] > 0), "Item, open item"])
private_events = dated_item_names("Open items", lambda oi: oi.loc[oi["Sales Category"] == "Event", "Open item"])
print(f"{shows.date.nunique()} ticketed show dates and {private_events.date.nunique()} private-event dates through {cal.date.max():%Y-%m-%d}")
private_events.sort_values("date").head(8).assign(name=lambda d: d["name"].map(redact))

In [ ]:
dc = pd.read_csv(CALENDAR_CSV, parse_dates=["start_date", "end_date"])
print(dc.kind.value_counts().to_string())
dc[["event", "kind", "start_date", "end_date", "confidence"]]

In [ ]:
def between_any(dates, rows):
    mask = pd.Series(False, index=dates.index)
    for r in rows.itertuples():
        mask |= dates.between(r.start_date, r.end_date)
    return mask


# Term in session = first day of classes through the end of finals (or last class day if finals unknown)
cal["term"] = "interim"
for r in dc[dc.kind == "term_classes"].itertuples():
    season, year = r.event.split()[:2]
    finals = dc[(dc.kind == "finals") & dc.event.str.contains(season) & dc.event.str.contains(year)]
    end = finals.end_date.max() if len(finals) else r.end_date
    cal.loc[cal.date.between(r.start_date, end), "term"] = season.lower()

BIG = ["commencement", "green_key", "homecoming", "winter_carnival", "family_weekend", "orientation"]
cal["big_event"] = between_any(cal.date, dc[dc.kind.isin(BIG)]).astype(int)
cal["football_home"] = between_any(cal.date, dc[dc.kind == "football_home"]).astype(int)
cal["show_night"] = cal.date.isin(shows.date).astype(int)
cal["private_event"] = cal.date.isin(private_events.date).astype(int)
for kind in BIG:
    cal[f"ev_{kind}"] = between_any(cal.date, dc[dc.kind == kind]).astype(int)
small_show = shows.name.str.upper().str.contains("COMEDY|IMPROV")
cal["show_comedy_improv"] = cal.date.isin(shows.loc[small_show, "date"]).astype(int)
cal["show_music"] = cal.date.isin(shows.loc[~small_show, "date"]).astype(int)
cal["week_of_term"] = -1   # weeks since the first day of classes; -1 between terms
for r in dc[dc.kind == "term_classes"].itertuples():
    season, year = r.event.split()[:2]
    fin = dc[(dc.kind == "finals") & dc.event.str.contains(season) & dc.event.str.contains(year)]
    span = cal.date.between(r.start_date, fin.end_date.max() if len(fin) else r.end_date)
    cal.loc[span, "week_of_term"] = (cal.loc[span, "date"] - r.start_date).dt.days // 7

# Holidays (section 8b). Rules, not a lookup table, so they extend to future dates.
from pandas.tseries.holiday import USFederalHolidayCalendar
FEDERAL = USFederalHolidayCalendar().holidays(cal.date.min(), cal.date.max(), return_name=True)
THANKSGIVING = FEDERAL[FEDERAL == "Thanksgiving Day"].index
YEARS = range(cal.date.min().year, cal.date.max().year + 1)
cal["hol_big_night"] = cal.date.isin(list(THANKSGIVING - pd.Timedelta(days=1))
                                     + [pd.Timestamp(y, 12, d) for y in YEARS for d in (23, 31)]).astype(int)
cal["hol_halloween"] = ((cal.date.dt.month == 10) & (cal.date.dt.day == 31)).astype(int)
cal["hol_closed"] = cal.date.isin(list(THANKSGIVING) + [pd.Timestamp(y, m, d) for y in YEARS
                                                        for m, d in ((1, 1), (7, 4), (12, 24), (12, 25))]).astype(int)

# Tested in section 8c, not adopted: Commencement week, and breaks split by season
cal["commencement_week"] = 0
for r in dc[dc.kind == "commencement"].itertuples():
    fin = dc[(dc.kind == "finals") & dc.event.str.contains(f"Spring {r.start_date.year}")].end_date.max()
    cal.loc[cal.date.between(fin + pd.Timedelta(days=1), r.start_date), "commencement_week"] = 1
BREAKS = {3: "spring_break", 6: "june_gap", 7: "june_gap", 8: "september_gap", 9: "september_gap",
          11: "winter_break", 12: "winter_break", 1: "winter_break"}
cal["term2"] = np.where(cal.term == "interim", cal.date.dt.month.map(BREAKS).fillna("other_break"), cal.term)
# Model v5 (section 9f): separate weekday levels after lunch service ended (assumed Jul 1, 2026; to confirm with Kieran)
LUNCH_END = pd.Timestamp("2026-07-01")
cal["weekday_regime"] = cal.weekday + np.where(cal.date >= LUNCH_END, " (no lunch)", "")
cal["per_order"] = cal.net_sales / cal.orders
cal["big_ticket_day"] = (cal.per_order >= 150).astype(int)
cal["is_2026"] = (cal.date.dt.year == 2026).astype(int)

od = cal[cal.net_sales.notna() & (cal.date <= "2026-10-21")].copy()   # days with numeric sales, up to the frozen forecasts' cutoff
# (DATA_THROUGH in section 9); later days are only for scoring in section 10. Closures are known from the schedule
od[["big_event", "football_home", "show_night", "private_event", "big_ticket_day"]].sum().rename("open days flagged").to_frame()

## 2. Model 1: daily demand

**Setup.** Median (quantile 0.5) linear regression on daily net sales. All inputs are known in advance, so each forecast is made once at the end of the training window and covers every day after it. That's a 1 to 113 day horizon, which includes the 7 and 28 day targets. No lagged sales are used.

**Validation.** The same three time-ordered folds as ChatGPT's frozen-block scorecard, so results compare row for row: 2025 H2 (trained on 2025 H1), 2026 H1 (trained on 2025), and 2026 H2 (Jul 1-Oct 21, trained on everything before). In each fold, a category never seen in training (for example the fall term when training on Jan-Jun 2025) falls back to the reference level, because nothing can be learned about it.

**Ablation.** Each row adds one data source. The drop in error is what that source is worth.

In [ ]:
FEATURES = {
    "M1  weekday + 2026 level": ["weekday", "is_2026"],
    "M2  + Dartmouth calendar": ["weekday", "is_2026", "term", "big_event", "football_home"],
    "M3  + Sawtooth show dates": ["weekday", "is_2026", "term", "big_event", "football_home", "show_night"],
    "M4  + big-ticket days (oracle)": ["weekday", "is_2026", "term", "big_event", "football_home", "show_night",
                                        "big_ticket_day"],
}
FOLDS = {  # name: (last training day, first test day, last test day)
    "2025 H2": ("2025-06-30", "2025-07-01", "2025-12-31"),
    "2026 H1": ("2025-12-31", "2026-01-01", "2026-06-30"),
    "2026 H2": ("2026-06-30", "2026-07-01", "2026-10-21"),
}


def design(cols, train_idx, drop_reference=True, frame=None):
    frame = od if frame is None else frame
    parts = []
    for c in cols:
        if c in ("weekday", "term", "term2", "weekday_regime"):
            dummies = pd.get_dummies(frame[c], prefix=c, dtype=float)
            reference = f"{c}_{frame.loc[train_idx, c].mode()[0]}"   # most common category in training
            parts.append(dummies.drop(columns=reference) if drop_reference else dummies)
        else:
            parts.append(frame[[c]].astype(float))
    X = pd.concat(parts, axis=1)
    return X.loc[:, X.loc[train_idx].nunique() > 1]   # nothing can be learned from a column constant in training


def fit_quantile(cols, train, q=0.5, frame=None, target="net_sales"):
    X = design(cols, train.index, frame=frame)
    model = QuantileRegressor(quantile=q, alpha=0.0, solver="highs").fit(X.loc[train.index], train[target])
    return model, X


def predict(cols, train, test, q=0.5):
    model, X = fit_quantile(cols, train, q)
    return pd.Series(model.predict(X.loc[test.index]), index=test.index)


def split(fold):
    train_end, test_start, test_end = FOLDS[fold]
    return od[od.date <= train_end], od[od.date.between(test_start, test_end)]


rows, preds = [], {}
for fold in FOLDS:
    train, test = split(fold)
    wd_median = train.groupby("weekday").net_sales.median()
    candidates = {"M0  weekday median (baseline)": test.weekday.map(wd_median)}
    candidates |= {name: predict(cols, train, test) for name, cols in FEATURES.items()}
    for name, p in candidates.items():
        preds[(fold, name)] = p
        rows.append({"fold": fold, "model": name, "days": len(test),
                     "abs_err": (test.net_sales - p).abs().sum(), "actual": test.net_sales.sum()})

ours = pd.DataFrame(rows)
pooled = ours.groupby("model")[["days", "abs_err", "actual"]].sum().assign(fold="ALL 3 FOLDS").reset_index()
ours = pd.concat([ours, pooled], ignore_index=True)
ours["WAPE"] = ours.abs_err / ours.actual
ours["MAE"] = ours.abs_err / ours.days
ours.pivot(index="model", columns="fold", values="WAPE")[list(FOLDS) + ["ALL 3 FOLDS"]].style.format("{:.1%}")

### Side by side with ChatGPT's scorecard

ChatGPT's numbers come from `data/chatgpt/Sawtooth_Model_Scorecard.csv`. Its "Weekday median" row should equal our M0, which checks that the folds line up. Its "1-day-ahead" rows use the previous day's actual sales, so they can't produce 7 or 28 day forecasts; they're shown for reference only.

In [ ]:
gpt = pd.read_csv("../data/chatgpt/Sawtooth_Model_Scorecard.csv")
gpt["fold"] = gpt.fold.str.replace(" (holdout)", "", regex=False)
gpt["model"] = np.where(gpt.protocol.str.startswith("Frozen"), "ChatGPT frozen: ", "ChatGPT 1-day-ahead: ") + gpt.model
gpt["WAPE"] = gpt["WAPE_%"] / 100
both = pd.concat([ours.assign(model="ours: " + ours.model)[["model", "fold", "WAPE"]], gpt[["model", "fold", "WAPE"]]])
table = both.pivot(index="model", columns="fold", values="WAPE")[list(FOLDS) + ["ALL 3 FOLDS"]]
table.sort_values("ALL 3 FOLDS").style.format("{:.1%}").highlight_min(axis=0, props="font-weight: bold")

### Are the forecast ranges calibrated?

Fit the 10th and 90th percentiles with the M3 inputs (everything we can know in advance today). A calibrated 80% range should contain about 80% of actual days.

In [ ]:
cov_rows, bands = [], {}
for fold in FOLDS:
    train, test = split(fold)
    lo = predict(FEATURES["M3  + Sawtooth show dates"], train, test, q=0.1)
    hi = predict(FEATURES["M3  + Sawtooth show dates"], train, test, q=0.9)
    bands[fold] = (lo, hi)
    cov_rows.append({"fold": fold, "coverage of 80% range": test.net_sales.between(lo, hi).mean(),
                     "median range width": (hi - lo).median(),
                     "actual above range": (test.net_sales > hi).mean(), "actual below range": (test.net_sales < lo).mean()})
pd.DataFrame(cov_rows).style.format({"coverage of 80% range": "{:.0%}", "median range width": "${:,.0f}",
                                     "actual above range": "{:.0%}", "actual below range": "{:.0%}"})

In [ ]:
train, test = split("2026 H2")
p = preds[("2026 H2", "M3  + Sawtooth show dates")]
band = bands["2026 H2"]

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.fill_between(test.date, band[0], band[1], color=BLUE_LIGHT, linewidth=0, label="80% forecast range")
ax.plot(test.date, p, color=BLUE, linewidth=2, label="Forecast (M3, made Jun 30)")
ax.scatter(test.date, test.net_sales, s=22, color=ORANGE, edgecolor=SURFACE, linewidth=1, zorder=3, label="Actual")
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"${v / 1000:.0f}K"))
ax.set_ylim(bottom=0)
ax.set_title("2026 H2: daily net sales vs forecast made on Jun 30, 2026")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncols=3)
plt.show()

### Where the remaining error is

The biggest misses in 2026 H2 for M3, with the flags each day had. Misses on days with no flag point at a data source the model doesn't have.

In [ ]:
miss = test.assign(forecast=p, error=test.net_sales - p)
miss["abs error"] = miss.error.abs()
cols = ["date", "weekday", "term", "net_sales", "forecast", "error", "orders", "per_order",
        "show_night", "big_event", "football_home", "big_ticket_day"]
miss.nlargest(12, "abs error")[cols].style.format(
    {"date": "{:%Y-%m-%d}", "net_sales": "${:,.0f}", "forecast": "${:,.0f}", "error": "${:+,.0f}",
     "orders": "{:.0f}", "per_order": "${:,.0f}"})

In [ ]:
# What M3 learned (trained through Jun 30, 2026). Effects are relative to the most common weekday and term.
model, X = fit_quantile(FEATURES["M3  + Sawtooth show dates"], od[od.date <= "2026-06-30"])
pd.Series(model.coef_, index=X.columns).round(0).sort_values().to_frame("effect on median daily sales ($)")

## 2b. Event days: baseline plus shrunken event effects

Fit a baseline for a normal day (weekday, term, 2026 level), then for each event type measure how far its past occurrences sat above that baseline, and apply that effect when the event comes up again. Effects are multiplicative, so a +20% Saturday and a +20% Tuesday differ in dollars.

Most event types have happened only once or twice, so a raw average is mostly noise. Each effect is shrunk toward zero by n / (n + λ), where n is the number of past days with that event. With λ = 10, an event seen twice keeps 1/6 of its measured effect and one seen 100 times keeps 91%. λ = 0 means no shrinkage.

Scored three ways, pooled over the three folds: all days; event days only (any Dartmouth event, ticketed show, or private-event date); and all days except big-ticket days.

In [ ]:
BASE = ["weekday", "is_2026", "term"]
DART_TYPES = [f"ev_{k}" for k in BIG] + ["football_home"]
EVENT_COLS = DART_TYPES + ["show_night", "private_event"]

register = pd.read_excel("../data/chatgpt/Sawtooth_Forecasting_Experiments_and_Missing_Data_Audit.xlsx",
                         sheet_name="EVENTS WITH SOURCE URL")
register = register[register["Default predictor eligibility"] == "YES"]
for t, g in register.groupby("Feature type"):
    od[f"gpt_{t}"] = od.date.isin(pd.to_datetime(g["Business date"])).astype(int)
GPT_TYPES = [c for c in od if c.startswith("gpt_")]


def two_stage(train, test, types, lam):
    model, X = fit_quantile(BASE, train)
    base_train = pd.Series(model.predict(X.loc[train.index]), index=train.index)
    base_test = pd.Series(model.predict(X.loc[test.index]), index=test.index)
    log_ratio = np.log(train.net_sales.clip(lower=50) / base_train.clip(lower=50))
    effects = {}
    for k in types:
        days = train.index[train[k] == 1]
        effects[k] = len(days) / (len(days) + lam) * log_ratio[days].median() if len(days) else 0.0
    return base_test * np.exp(sum(test[k] * effects[k] for k in types)), effects


M2 = FEATURES["M2  + Dartmouth calendar"]
VARIANTS = {
    "M0 weekday median": lambda tr, te: te.weekday.map(tr.groupby("weekday").net_sales.median()),
    "M2 (grouped event flags)": lambda tr, te: predict(M2, tr, te),
    "Baseline only, no events": lambda tr, te: predict(BASE, tr, te),
    "M2 + private-event dates": lambda tr, te: predict(M2 + ["private_event"], tr, te),
    "Two-stage, Dartmouth types, λ=0": lambda tr, te: two_stage(tr, te, DART_TYPES, 0)[0],
    "Two-stage, Dartmouth types, λ=10": lambda tr, te: two_stage(tr, te, DART_TYPES, 10)[0],
    "Two-stage, + shows + private-event dates, λ=10": lambda tr, te: two_stage(tr, te, EVENT_COLS, 10)[0],
    "Two-stage, ChatGPT register types, λ=10": lambda tr, te: two_stage(tr, te, GPT_TYPES, 10)[0],
}
rows = []
for fold in FOLDS:
    train, test = split(fold)
    on_event, normal = test[EVENT_COLS].max(axis=1) == 1, test.big_ticket_day == 0
    for name, f in VARIANTS.items():
        err = (test.net_sales - f(train, test)).abs()
        rows.append({"model": name, "err": err.sum(), "act": test.net_sales.sum(),
                     "err_event": err[on_event].sum(), "act_event": test.net_sales[on_event].sum(),
                     "err_normal": err[normal].sum(), "act_normal": test.net_sales[normal].sum(),
                     "event days": int(on_event.sum())})
g = pd.DataFrame(rows).groupby("model", sort=False).sum()
pd.DataFrame({"all days": g.err / g.act, "event days only": g.err_event / g.act_event,
              "excluding big-ticket days": g.err_normal / g.act_normal, "event days scored": g["event days"]}).style.format(
    {"all days": "{:.1%}", "event days only": "{:.1%}", "excluding big-ticket days": "{:.1%}", "event days scored": "{:.0f}"}
).highlight_min(subset=["all days", "event days only", "excluding big-ticket days"], props="font-weight: bold")

What the two-stage model learned (trained through Jun 30, 2026), before and after shrinkage:

In [ ]:
train = od[od.date <= "2026-06-30"]
_, raw = two_stage(train, train.iloc[:1], EVENT_COLS, 0)
_, shrunk = two_stage(train, train.iloc[:1], EVENT_COLS, 10)
pd.DataFrame({"past days": {k: int(train[k].sum()) for k in EVENT_COLS},
              "raw effect": {k: np.expm1(v) for k, v in raw.items()},
              "shrunk effect (λ=10)": {k: np.expm1(v) for k, v in shrunk.items()}}).sort_values("raw effect").style.format(
    {"raw effect": "{:+.0%}", "shrunk effect (λ=10)": "{:+.0%}"})

**Takeaway.** Shrinkage works as intended. Without it (λ=0), event effects make forecasts worse than ignoring events (39.8% vs 39.0%). With λ=10 they help slightly on event days (36.2% vs 36.5%). The gain is small because each Dartmouth event has only 1 to 11 past days, so even large raw effects (Family Weekend +68%, Green Key -37%) shrink to around ±10%. ChatGPT's 797-row register makes things worse even with shrinkage (40.1%).

The one event source that clearly helps is **private-event dates**: M2 plus those dates is the best model here (38.7% on all days, 36.0% on event days), and ours is only a partial list rebuilt from payment names. Getting event days materially better needs three things: the full bookings calendar with headcounts, more years of history so each Dartmouth event has more than one or two past occurrences, and Event sales by day so payments can be separated from parties.

λ was compared on these same test periods, so treat the λ=10 rows as slightly optimistic.

## 2c. Mapping days to specific events

Three ways to go beyond event *types*:

1. **Which show, not just "a show".** Comedy and improv nights sell a median of 12-18 tickets; band nights 70-110. Split the show flag in two.
2. **Last year's occurrence of the same event.** Forecast each 2026 event day from the same event's 2025 day with the same weekday, scaled by how that weekday's sales changed year over year (median of the 8 weeks ending a week before, against the same 8 weeks a year earlier).
3. **Every day mapped to the same weekday last year**, scaled the same way, as a 7 or 28 day ahead forecast.

In [ ]:
M2P = FEATURES["M2  + Dartmouth calendar"] + ["private_event"]
SHOW_VARIANTS = {"M2 + private-event dates": M2P,
                 "  + one show flag": M2P + ["show_night"],
                 "  + comedy/improv and band flags": M2P + ["show_comedy_improv", "show_music"]}
rows, best_preds = [], {}
for fold in FOLDS:
    train, test = split(fold)
    for name, cols in SHOW_VARIANTS.items():
        p = predict(cols, train, test)
        if name == "M2 + private-event dates":
            best_preds[fold] = p
        err = (test.net_sales - p).abs()
        for subset, mask in [("all days", test.net_sales.notna()), ("band nights", test.show_music == 1),
                             ("comedy/improv nights", test.show_comedy_improv == 1)]:
            rows.append({"model": name, "subset": subset, "err": err[mask].sum(), "act": test.net_sales[mask].sum()})
g = pd.DataFrame(rows).groupby(["model", "subset"], sort=False).sum()
print("Shows by weekday:", od.loc[od.show_night == 1, "weekday"].value_counts().to_dict())
(g.err / g.act).unstack().style.format("{:.1%}")

In [ ]:
SALES = od.set_index("date").net_sales


def yoy_level(day, lag=7, weeks=8):
    # Same-weekday median over `weeks` weeks ending `lag` days before `day`, against the same window a year earlier.
    same = SALES[SALES.index.day_of_week == day.day_of_week]
    now = same[(same.index > day - pd.Timedelta(days=lag + 7 * weeks)) & (same.index <= day - pd.Timedelta(days=lag))]
    then = same[(same.index > day - pd.Timedelta(days=lag + 7 * weeks + 364)) & (same.index <= day - pd.Timedelta(days=lag + 364))]
    return now.median() / then.median() if len(now) and len(then) else np.nan


rows = []
for kind in BIG + ["football_home"]:
    col = kind if kind == "football_home" else f"ev_{kind}"
    last_year = od[(od[col] == 1) & (od.date.dt.year == 2025)]
    for r in od[(od[col] == 1) & (od.date.dt.year == 2026)].itertuples():
        match = last_year[last_year.weekday == r.weekday]
        if match.empty:
            continue
        fold = "2026 H1" if r.date <= pd.Timestamp("2026-06-30") else "2026 H2"
        rows.append({"event": kind, "date": r.date, "weekday": r.weekday, "actual": r.net_sales,
                     "last year": match.net_sales.median(), "level ratio": yoy_level(r.date),
                     "model": best_preds[fold][r.Index]})
analog = pd.DataFrame(rows).drop_duplicates("date")   # a day can be both Family Weekend and a home game
analog["last year x level"] = analog["last year"] * analog["level ratio"]
analog["50/50 blend"] = (analog["last year x level"] + analog.model) / 2
wape = lambda col: (analog.actual - analog[col]).abs().sum() / analog.actual.sum()
display(pd.Series({c: wape(c) for c in ["last year x level", "model", "50/50 blend"]},
                  name=f"WAPE on {len(analog)} 2026 event days").to_frame().style.format("{:.1%}"))
analog.style.format({"date": "{:%Y-%m-%d}", "actual": "${:,.0f}", "last year": "${:,.0f}", "level ratio": "{:.2f}",
                     "model": "${:,.0f}", "last year x level": "${:,.0f}", "50/50 blend": "${:,.0f}"}).hide(axis="index")

In [ ]:
rows = []
for fold in ["2026 H1", "2026 H2"]:
    train, test = split(fold)
    actual = test.set_index("date").net_sales
    model = pd.Series(best_preds[fold].values, index=test.date)
    for lag in (7, 28):
        yoy = pd.Series({day: SALES.get(day - pd.Timedelta(days=364), np.nan) * yoy_level(day, lag) for day in actual.index})
        ok = yoy.notna()
        w = lambda f: (actual[ok] - f[ok]).abs().sum() / actual[ok].sum()
        rows.append({"fold": fold, "days ahead": lag, "days scored": int(ok.sum()),
                     "same weekday last year x level": w(yoy), "model (made months ahead)": w(model),
                     "50/50 blend": w((yoy + model) / 2)})
pd.DataFrame(rows).style.format({c: "{:.1%}" for c in ["same weekday last year x level", "model (made months ahead)",
                                                       "50/50 blend"]}).hide(axis="index")

**Takeaway.**

- Splitting shows by act type doesn't help total sales (39.2% vs 38.7%, and band nights get worse). Shows run on about half of all Wednesdays and Fridays, so the weekday effect already absorbs them. They may still change the drink mix, which needs category sales by day to test.
- Last year's same event does about as well as the model on its own (25.7% vs 23.7% on 19 event days). The 50/50 blend is clearly better (19.8%) because the two miss on different days. It's only 19 days and the blend was picked after seeing them, so treat it as promising, not proven.
- Mapping every day to the same weekday last year is far worse (56% to 72%): one past day is too noisy. Last year only helps where the event itself repeats.

## 2d. XGBoost with the engineered features

Gradient-boosted trees find interactions on their own (a Friday during Green Key) and handle missing values, but they can't extrapolate a level shift and overfit easily on 548 rows. The settings were fixed before looking at any results: depth-3 trees, 400 rounds at learning rate 0.03, row and column subsampling, an L2 penalty, and absolute-error loss so it targets the median like the other models.

Three ways to use it:
- **XGBoost on sales**, with the same inputs as the best linear model.
- **XGBoost on sales**, with every engineered input: each Dartmouth event type, private-event dates, comedy vs band shows, and week of term.
- **Hybrid.** The linear median model sets the baseline (weekday, term, 2026 level), and XGBoost learns only how events and week of term move sales relative to it (log ratio to the baseline).

Event days are the same set as in 2b.

In [ ]:
import xgboost as xgb

XGB_PARAMS = dict(objective="reg:absoluteerror", n_estimators=400, learning_rate=0.03, max_depth=3, min_child_weight=5,
                  subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, random_state=0, n_jobs=1)
EVENT_INPUTS = [f"ev_{k}" for k in BIG] + ["football_home", "private_event", "show_comedy_improv", "show_music"]
ALL_INPUTS = BASE + EVENT_INPUTS + ["week_of_term"]


def xgb_fit(cols, train, target, frame=None):
    X = design(cols, train.index, drop_reference=False, frame=frame)
    return xgb.XGBRegressor(**XGB_PARAMS).fit(X.loc[train.index], target), X


def xgb_predict(cols, train, test):
    model, X = xgb_fit(cols, train, train.net_sales)
    return pd.Series(model.predict(X.loc[test.index]), index=test.index)


def hybrid(train, test):
    base, Xb = fit_quantile(BASE, train)
    base_train = pd.Series(base.predict(Xb.loc[train.index]), index=train.index)
    base_test = pd.Series(base.predict(Xb.loc[test.index]), index=test.index)
    resid = np.log(train.net_sales.clip(lower=50) / base_train.clip(lower=50))
    model, X = xgb_fit(EVENT_INPUTS + ["week_of_term", "weekday"], train, resid)
    return base_test * np.exp(model.predict(X.loc[test.index])), model


XGB_VARIANTS = {
    "Linear: M2 + private-event dates (best so far)": lambda tr, te: predict(M2P, tr, te),
    "Linear: every engineered input": lambda tr, te: predict(ALL_INPUTS, tr, te),
    "XGB: same inputs as best linear": lambda tr, te: xgb_predict(M2P, tr, te),
    "XGB: every engineered input": lambda tr, te: xgb_predict(ALL_INPUTS, tr, te),
    "Hybrid: linear baseline + XGB event effects": lambda tr, te: hybrid(tr, te)[0],
}
rows = []
for fold in FOLDS:
    train, test = split(fold)
    on_event, normal = test[EVENT_COLS].max(axis=1) == 1, test.big_ticket_day == 0
    for name, f in XGB_VARIANTS.items():
        err = (test.net_sales - f(train, test)).abs()
        rows.append({"model": name, "fold": fold, "err": err.sum(), "act": test.net_sales.sum(),
                     "err_event": err[on_event].sum(), "act_event": test.net_sales[on_event].sum(),
                     "err_normal": err[normal].sum(), "act_normal": test.net_sales[normal].sum()})
r = pd.DataFrame(rows)
per_fold = r.assign(w=r.err / r.act).pivot(index="model", columns="fold", values="w").loc[list(XGB_VARIANTS)]
g = r.groupby("model", sort=False)[["err", "act", "err_event", "act_event", "err_normal", "act_normal"]].sum()
per_fold.assign(**{"all 3 folds": g.err / g.act, "event days only": g.err_event / g.act_event,
                   "excluding big-ticket days": g.err_normal / g.act_normal}).style.format("{:.1%}").highlight_min(
    axis=0, props="font-weight: bold")

In [ ]:
# What the hybrid's trees lean on (trained through Jun 30, 2026)
_, model = hybrid(od[od.date <= "2026-06-30"], od.iloc[:1])
gain = pd.Series(model.get_booster().get_score(importance_type="gain"))
(gain / gain.sum()).sort_values(ascending=False).head(12).to_frame("share of total gain").style.format("{:.0%}")

**Takeaway.** XGBoost on its own is worse than the linear median model overall (39.6% to 39.8% vs 38.7%), as expected with 548 rows and a level shift it can't extrapolate. It does better on event days (34.8% to 35.1% vs 36.0%), which is where interactions matter.

The hybrid gets the best of both. The linear model handles weekday, term and the year level, and the trees handle events. It's the best model in this notebook: 38.0% overall, 34.0% on event days, 34.7% excluding big-ticket days. It beats the linear model in two of the three periods (slightly worse in 2026 H1). The trees lean most on private-event dates, week of term, comedy/improv nights and Orientation.

The gain over the linear model is small (0.7 points), and three XGBoost designs were compared on these same periods, so some of it may be luck. A prospective test (forecast November now, score it in December) would settle it. The hybrid doesn't produce forecast ranges yet; M3's 80% ranges remain the calibrated reference.

## 3. Model 2: night types (rules, not ML)

The status update's three night types, plus big-ticket days (private parties or invoice payments), labeled with the sources we have. Precedence: big-ticket day, then Sawtooth show, then Dartmouth event, then regular.

In [ ]:
od["night type"] = np.select(
    [od.big_ticket_day == 1, od.show_night == 1, (od.big_event == 1) | (od.football_home == 1)],
    ["big-ticket day (party or payment)", "Sawtooth show", "Dartmouth event"], "regular")
od.groupby("night type").agg(days=("date", "size"), median_sales=("net_sales", "median"),
                             median_orders=("orders", "median"), median_per_order=("per_order", "median")).style.format(
    {"median_sales": "${:,.0f}", "median_orders": "{:.0f}", "median_per_order": "${:,.0f}"})

### How private-event money shows up in Toast

The "Event" sales category is mostly deposits, installments, remainders and invoices for private parties, formals and catering, typed in as open items. Each is posted on the day it's **paid**, which can be weeks from the event (the Jul-Oct 2026 export already holds a deposit for an 11/12 party). So some revenue spikes are payments rather than parties, and separating the two needs Event (or all-category) sales by day. The event dates written in those names do line up with busy days, though (next cell), so a bookings calendar with event dates and headcounts is worth asking for.

In [ ]:
ev_rows = []
for p in sorted(DATA.glob("ProductMix_202[56]-*.xlsx")):
    oi = pd.read_excel(p, sheet_name="Open items")
    ev = oi[oi["Sales Category"] == "Event"]
    names = ev["Open item"].astype(str)
    ev_rows.append({
        "export": p.stem.replace("ProductMix_", ""), "Event open items": len(ev), "Event net $": ev["Net item amt"].sum(),
        "deposit / invoice / installment / remainder / payment": int(names.str.contains(
            "deposit|invoice|installment|remainder|payment", case=False).sum()),
        "name contains a date": int(names.str.contains(r"\b\d{1,2}[/\\]\d{1,2}\b").sum()),
        "three largest": " | ".join(ev.nlargest(3, "Net item amt")["Open item"].map(redact)),
    })
pd.DataFrame(ev_rows).style.format({"Event net $": "${:,.0f}"})

In [ ]:
wd_median = od.groupby("weekday").net_sales.median()
ratio = od.net_sales / od.weekday.map(wd_median)
named = od.private_event == 1
display(pd.DataFrame({
    "days": [int(named.sum()), len(od)],
    "median sales vs normal for that weekday": [ratio[named].median(), ratio.median()],
    "share above 1.5x normal": [(ratio[named] > 1.5).mean(), (ratio > 1.5).mean()],
}, index=["named private-event dates", "all open days"]).style.format(
    {"median sales vs normal for that weekday": "{:.2f}x", "share above 1.5x normal": "{:.0%}"}))
pd.cut(od.loc[od.big_ticket_day == 1, "orders"], [0, 5, 40, 10_000],
       labels=["1-5 orders (payment-like)", "6-40 orders", "41+ orders"]).value_counts().sort_index().rename("big-ticket days").to_frame()

## 4. Model 3: drink mix (can't train yet)

The only category split is per export period, so there are four data points. Drink share of net sales is moving fast, so a fixed ratio would be wrong too.

In [ ]:
PERIOD = {
    "sales-breakdown-jan1-jun30-2025-2.csv": "2025 H1",
    "sales-breakdown-jul1-dec31-2025.csv": "2025 H2",
    "sales-breakdown-jan1-jun30-2026.csv": "2026 H1",
    "sales-breakdown-jul1-oct21-2026.csv": "2026 Jul 1-Oct 21",
}
BEV = ["Liquor", "Draft Beer", "Bottled Beer", "Wine", "NA Beverage"]
mix = {}
for f, label in PERIOD.items():
    df = pd.read_csv(DATA / f, index_col=False)
    tot = df[df["Service Type"].isna()].set_index("Sales Category")["Net Sales"]
    mix[label] = (tot[BEV] / tot.sum()).to_dict() | {"all drinks": tot[BEV].sum() / tot.sum(), "drinks net $": tot[BEV].sum()}
mix = pd.DataFrame(mix).T
mix.style.format({c: "{:.1%}" for c in BEV + ["all drinks"]} | {"drinks net $": "${:,.0f}"})

## 5. Model 4: reorder points and order quantities (can't build yet)

This is inventory math, not ML: reorder when stock falls below expected demand over the delivery lead time plus a safety margin taken from the forecast range. It needs inputs we don't have.

In [ ]:
pd.DataFrame([
    ("Drink demand forecast by category or item", "No", "Model 3 (needs daily or weekly category sales)"),
    ("Pour size or recipe per drink (oz per serving)", "No", "Kieran / bar manager"),
    ("Case or keg size and unit cost per product", "No", "Distributor invoices"),
    ("Delivery lead time and order days per distributor", "No", "Kieran"),
    ("Current on-hand counts (or a recent inventory count)", "No", "Kieran"),
    ("Stockout history (what ran out, when)", "No", "Kieran; also answers Laura's question"),
], columns=["input", "have it?", "source"])

## 6. Checking the email's request: two Product Mix weekends

ChatGPT's email asked for Product Mix exports for Apr 11-12, 2025 (comparison) and May 16-17, 2025 (Green Key, with a Sawtooth show on May 17). The question is whether one event weekend against one comparison weekend can show an event effect. Below: Friday plus Saturday net sales for every spring-term weekend in both years, with Green Key and the email's comparison weekend marked.

In [ ]:
fs = cal[cal.date.dt.day_of_week.isin([4, 5]) & (cal.term == "spring")]
wk = fs.groupby(fs.date.dt.to_period("W-SUN")).agg(fri=("date", "min"), days=("date", "size"),
                                                   fri_sat_sales=("net_sales", "sum"), shows=("show_night", "sum"))
wk = wk[wk.days == 2].copy()
wk["year"] = wk.fri.dt.year
green_key_fridays = set(dc.loc[dc.kind == "green_key", "start_date"])
wk["weekend"] = np.select([wk.fri.isin(green_key_fridays), wk.fri == pd.Timestamp("2025-04-11")],
                          ["Green Key", "email's comparison"], "")
wk["rank in its spring (1 = highest)"] = wk.groupby("year").fri_sat_sales.rank(ascending=False).astype(int)
wk["of"] = wk.groupby("year").fri.transform("size")
wk[["year", "fri", "fri_sat_sales", "shows", "weekend", "rank in its spring (1 = highest)", "of"]].style.format(
    {"fri": "{:%Y-%m-%d}", "fri_sat_sales": "${:,.0f}", "shows": "{:.0f}"})

## 7. What to ask Kieran for

Ranked for the project's stated goal (Written Status Update 1): forecast drink demand and set reorder points and order quantities for bar inventory. "Unblocks" says which of the four models each ask feeds.

| Priority | Ask | Unblocks | Evidence | In ChatGPT's email? |
|---|---|---|---|---|
| 1 | Sales by category (liquor, draft, bottled, wine, NA, Event) per day or per week, if Toast has a report for it. If it only exports one date range at a time, monthly Sales Summaries for 2025-2026 instead of doing daily or weekly by hand | 3, 4; also cleans 1 | Drinks went from 36% to 56% of net sales, so total sales isn't drink demand. Event payments (deposits, invoices) land on payment days; separating them from parties needs Event sales by day | Buried in a parenthetical under the Event question |
| 2 | For the top ~20 drinks (we can send the list): pour size, case or keg size and cost, order days and delivery time, a recent count, and what ran out and when | 4 | No inventory inputs exist at all. Stockout history also answers Laura's question | No |
| 3 | A calendar of private events and Sawtooth shows, including unticketed DJ or dance nights: date, headcount, and whether drinks were included | 1, 2, 4 | Named private-event dates run 1.41x a normal day, and adding even our partial list gives the best model (section 2b). The Aug 22, 2026 spike ($11.6K, DJ and dance nights) had no ticket. We can send our partial list so Kieran only fills gaps | Partly (as "bookings and shows", only if readily available) |
| 4 | Sales Summary exports for 2022-2024 (the same export as before) | 1 | Each Dartmouth event has only one or two past occurrences, so large raw effects (Family Weekend +68%) shrink to about ±10% (section 2b). Toast data goes back at least to Sep 2022 | No |
| 5 | Anything else that changed in the last year or so: kitchen, menu, prices | 1 (calibration) | Net sales -22.6% year over year, food -29% (H1 vs H1), net per order -20%. The 2026 H1 forecast ranges missed low on 23% of days | Hours yes; the rest no |
| Done | What the "Event" category is | Cleaning | Mostly answered by the data: deposits, invoices and installments for private events (section 3) | Yes |
| Low | Product Mix for two weekends (Apr 11-12 and May 16-17, 2025) | None new | Custom date ranges already work (the 2022 export covers Sep 11-Oct 31). Green Key 2025 was the lowest spring weekend and Green Key 2026 the 3rd highest, so one pair can't show an event effect | Yes, as the main ask |

For reference, ChatGPT's own ranked list (`data/chatgpt/Sawtooth_Prioritized_Missing_Data.csv`):

In [ ]:
pd.read_csv("../data/chatgpt/Sawtooth_Prioritized_Missing_Data.csv")[["Rank", "Missing data", "Minimal field request", "Owner effort"]]

## 8. Rolling backtest

The frozen backtest above refits once per six-month period. In practice you'd refit regularly and forecast the next week or month. Here every model is refit at the end of each period on all data to date, starting after the first full year (Dec 31, 2025), so every term has been seen once. Three refit schedules are compared: every 6 months, every 3 months, and every month.

The models use only inputs known in advance, so a forecast's horizon matters only through which refit it can use. A forecast made 28 days ahead must come from a refit at least 28 days before the day (or before the Monday of its week, for weekly totals). Scored on Feb 2 to Oct 18, 2026 (37 full weeks), where every schedule and horizon has a usable refit.

**80% ranges.** Daily ranges come from the hybrid's 10th and 90th percentile models. Summing those into weeks would be far too wide (daily misses partly cancel), so weekly ranges come from the hybrid's own past weekly errors: the 10th to 90th percentile of errors on weeks that had already finished when the forecast was made (at least 8 needed).

In [ ]:
FLOOR = {"net_sales": 50, "orders": 1, "guests": 1}   # keeps log ratios finite on near-empty days


def hybrid_fit(train, frame=None, target="net_sales", base_cols=None, event_cols=None):
    # Linear quantile baselines (10th, 50th, 90th percentile) times one XGBoost event multiplier.
    base_cols = BASE if base_cols is None else base_cols
    event_cols = EVENT_INPUTS + ["week_of_term", "weekday"] if event_cols is None else event_cols
    base = {q: fit_quantile(base_cols, train, q, frame=frame, target=target) for q in (0.1, 0.5, 0.9)}
    model, X = base[0.5]
    base_train = pd.Series(model.predict(X.loc[train.index]), index=train.index)
    f = FLOOR[target]
    resid = np.log(train[target].clip(lower=f) / base_train.clip(lower=f))
    trees, X_trees = xgb_fit(event_cols, train, resid, frame=frame)
    return base, trees, X_trees


def hybrid_predict(fit, rows):
    base, trees, X_trees = fit
    multiplier = np.exp(trees.predict(X_trees.loc[rows.index]))
    return {q: pd.Series(m.predict(X.loc[rows.index]) * multiplier, index=rows.index) for q, (m, X) in base.items()}


REFITS = pd.date_range("2025-12-31", "2026-09-30", freq="ME")
SCHEDULES = {"every 6 months": REFITS[[0, 6]], "every 3 months": REFITS[[0, 3, 6, 9]], "every month": REFITS}
ROLL_END = pd.Timestamp("2026-10-21")
SCORE_START, SCORE_END = pd.Timestamp("2026-02-02"), pd.Timestamp("2026-10-18")
MODELS = ["weekday median", "linear", "hybrid"]

pieces = []
for r in REFITS:
    train, rows = od[od.date <= r], od[(od.date > r) & (od.date <= ROLL_END)]
    h = hybrid_predict(hybrid_fit(train), rows)
    pieces.append(pd.DataFrame({
        "refit": r, "date": rows.date, "actual": rows.net_sales,
        "weekday median": rows.weekday.map(train.groupby("weekday").net_sales.median()),
        "linear": predict(M2P, train, rows), "hybrid": h[0.5], "hybrid_lo": h[0.1], "hybrid_hi": h[0.9]}))
roll = pd.concat(pieces, ignore_index=True)


def pick(schedule, horizon, weekly):
    # For each day, the forecast from the latest scheduled refit made at least `horizon` days
    # before the day (or before the Monday of its week, for weekly totals).
    p = roll[roll.refit.isin(SCHEDULES[schedule])]
    anchor = p.date - pd.to_timedelta(p.date.dt.day_of_week, unit="D") if weekly else p.date
    p = p[p.refit <= anchor - pd.Timedelta(days=horizon)]
    p = p.sort_values("refit").groupby("date").tail(1)
    return p[p.date.between(SCORE_START, SCORE_END)]


acc_rows, cal_rows = [], []
for schedule in SCHEDULES:
    for horizon in (7, 28):
        dp, wp = pick(schedule, horizon, weekly=False), pick(schedule, horizon, weekly=True)
        weeks = wp.groupby(wp.date.dt.to_period("W-SUN"))[["actual"] + MODELS].sum()
        row = {"refit": schedule, "days ahead": horizon}
        for m in MODELS:
            row[f"daily: {m}"] = (dp.actual - dp[m]).abs().sum() / dp.actual.sum()
        for m in MODELS:
            row[f"weekly: {m}"] = (weeks.actual - weeks[m]).abs().sum() / weeks.actual.sum()
        acc_rows.append(row)
        log_err = np.log(weeks.actual / weeks.hybrid)
        ends, starts = weeks.index.end_time.normalize(), weeks.index.start_time
        inside = []
        for i in range(len(weeks)):
            past = log_err[ends <= starts[i] - pd.Timedelta(days=horizon)]
            if len(past) >= 8:
                lo, hi = np.quantile(past, [0.1, 0.9])
                inside.append(lo <= log_err.iloc[i] <= hi)
        cal_rows.append({"refit": schedule, "days ahead": horizon,
                         "daily 80% range coverage": dp.actual.between(dp.hybrid_lo, dp.hybrid_hi).mean(),
                         "weekly 80% range coverage": np.mean(inside) if inside else np.nan, "weeks with a range": len(inside)})

acc = pd.DataFrame(acc_rows)
display(acc.style.format({c: "{:.1%}" for c in acc.columns if ":" in c}).hide(axis="index").highlight_min(
    subset=[c for c in acc.columns if ":" in c], props="font-weight: bold"))
pd.DataFrame(cal_rows).style.format({"daily 80% range coverage": "{:.0%}", "weekly 80% range coverage": "{:.0%}"}).hide(axis="index")

**Takeaway.**

- **Refit cadence barely matters.** Refitting every 3 months is within about a point of refitting monthly on every measure, so quarterly refits are enough. Monthly is slightly better on daily error (34.1-34.7% vs 35.0-35.2%).
- **Horizon barely matters either.** 28 days ahead is within about a point of 7 days ahead, as expected for a model that doesn't use recent sales.
- **Weekly error is about 18-19% for the hybrid** with quarterly refits, against 23-24% for the weekday median. Daily, the hybrid and the linear model are tied (about 35%); weekly, the hybrid is 0.6-1.2 points better.
- **The 80% ranges run a little narrow.** They contained 73-77% of days and 68-80% of weeks. With only 25-28 weeks scored, a few points either way is noise, but the lean is consistently below 80%, so read them as roughly 75% ranges.
- These scores (Feb-Oct 2026, with at least a year of history) aren't comparable to the frozen backtest's 38%, which covers different periods, including 2025 H2 when only six months of history existed.

## 8b. Holidays (model v2)

Each holiday has happened once or twice in this data, so per-holiday effects can't be learned. Holidays that behave alike are grouped so a group's effect is learned from several days. To pick the groups, every holiday is compared with the median of the same weekday in the surrounding four weeks (excluding other holidays):

In [ ]:
named = {}
for d, name in FEDERAL.items():
    named.setdefault(d, []).append(f"federal: {name}")
for y in YEARS:
    for (m, day), name in {(10, 31): "Halloween", (3, 17): "St. Patrick's Day", (2, 14): "Valentine's Day",
                           (12, 23): "Dec 23", (12, 31): "New Year's Eve"}.items():
        named.setdefault(pd.Timestamp(y, m, day), []).append(f"night out: {name}")
for t in THANKSGIVING:
    named.setdefault(t - pd.Timedelta(days=1), []).append("night out: Thanksgiving Eve")
    named.setdefault(t + pd.Timedelta(days=1), []).append("home for holidays: day after Thanksgiving")
    named.setdefault(t + pd.Timedelta(days=2), []).append("home for holidays: Saturday after Thanksgiving")
for y in YEARS:
    for d in pd.date_range(f"{y}-12-26", f"{y}-12-30"):
        named.setdefault(d, []).append("home for holidays: Dec 26-30")
hs = od[od.date <= "2026-10-21"].set_index("date").net_sales
rows = []
for d, names in sorted(named.items()):
    if d > hs.index.max():
        continue
    near = hs[(hs.index.day_of_week == d.day_of_week) & (abs((hs.index - d).days) <= 28) & ~hs.index.isin(list(named))]
    rows.append({"date": d, "holiday": "; ".join(names), "sales": hs.get(d, np.nan),
                 "same weekday nearby": near.median(), "ratio": hs.get(d, np.nan) / near.median()})
hol = pd.DataFrame(rows)
hol["group"] = hol.holiday.str.split(":").str[0]
display(hol.dropna(subset=["sales"]).groupby("group").ratio.agg(["count", "median", "min", "max"]).style.format(
    {"median": "{:.2f}", "min": "{:.2f}", "max": "{:.2f}"}))
hol.style.format({"date": "{:%Y-%m-%d}", "sales": "${:,.0f}", "same weekday nearby": "${:,.0f}", "ratio": "{:.2f}"},
                 na_rep="closed").hide(axis="index")

**What v2 adds.** Three of the "night out" days stand far apart from the rest: Thanksgiving Eve (3.0x), Dec 23 (4.1x) and New Year's Eve (4.2x), the nights when locals are home and students are gone. They become one group, `hol_big_night`. Halloween (1.5x, once) gets its own flag. St. Patrick's, Valentine's, the days after Thanksgiving, Dec 26-30 and the federal holidays show no consistent effect, so they're left out. Monday holidays ran about 1.5x a normal Monday, but Sawtooth now closes Mondays. New Year's Day, July 4, Thanksgiving, Dec 24 and Dec 25 were effectively closed, so they go into the closure schedule (`hol_closed`) instead of the demand model.

**How.** v2 is the hybrid times a holiday multiplier. Each group's effect is the median log ratio of actual sales to the hybrid's own fit on that group's past days, shrunk by n / (n + λ). For big nights, λ is chosen by leave-one-out on the three past big nights (each predicted from the other two), using training data only. Halloween has been seen once, so there's nothing to check it against; it keeps λ = 3, a quarter of its measured effect.

**Why there's no backtest.** The three big nights all fall in Nov-Dec 2025. The only test period containing them (2025 H2) trains on Jan-Jun 2025, which has none, so v2 equals v1 in every backtest above. The leave-one-out below is all the history allows, and it's also what picks λ. Three points is a fragile basis for that choice; the real test is the frozen v2 forecast in section 9.

In [ ]:
LAMBDAS = [0, 1, 3, 10]


def holiday_effects(train, fit, lams, target="net_sales"):
    # Per holiday group: shrunken median log ratio of actual values to the hybrid's in-sample forecast.
    pred = hybrid_predict(fit, train)[0.5]
    f = FLOOR[target]
    log_ratio = np.log(train[target].clip(lower=f) / pred.clip(lower=f))
    effects = {}
    for k, lam in lams.items():
        days = train.index[train[k] == 1]
        effects[k] = len(days) / (len(days) + lam) * log_ratio[days].median() if len(days) else 0.0
    return effects


def holiday_multiplier(rows, effects):
    return np.exp(sum(rows[k] * v for k, v in effects.items()))


hist = od[od.date <= "2026-10-21"]
loo = []
for idx in hist.index[hist.hol_big_night == 1]:
    tr = hist.drop(index=idx)
    fit = hybrid_fit(tr)
    one = hist.loc[[idx]]
    v1 = hybrid_predict(fit, one)[0.5].iloc[0]
    row = {"date": one.date.iloc[0], "actual": one.net_sales.iloc[0], "v1 (no holidays)": v1}
    for lam in LAMBDAS:
        row[f"v2, λ={lam}"] = v1 * holiday_multiplier(one, holiday_effects(tr, fit, {"hol_big_night": lam})).iloc[0]
    loo.append(row)
loo = pd.DataFrame(loo)
pred_cols = [c for c in loo.columns if c.startswith("v")]
loo_wape = pd.Series({c: (loo.actual - loo[c]).abs().sum() / loo.actual.sum() for c in pred_cols},
                     name="WAPE on the 3 held-out big nights")
HOLIDAY_LAMBDAS = {"hol_big_night": min(LAMBDAS, key=lambda lam: loo_wape[f"v2, λ={lam}"]), "hol_halloween": 3}
display(loo.style.format({"date": "{:%Y-%m-%d}"} | {c: "${:,.0f}" for c in ["actual"] + pred_cols}).hide(axis="index"))
display(loo_wape.to_frame().style.format("{:.1%}"))
print("Chosen λ:", HOLIDAY_LAMBDAS)

full_fit = hybrid_fit(hist)
eff = holiday_effects(hist, full_fit, HOLIDAY_LAMBDAS)
pd.Series({k: np.expm1(v) for k, v in eff.items()}, name="v2 holiday effect (trained through Oct 21, 2026)").to_frame().style.format("{:+.0%}")

**Takeaway.** Leave-one-out picked no shrinkage (λ = 0) for big nights. Predicting each of the three from the other two, v2 is off by 14.7% in total, against 67.2% for v1 and 50.4% with λ = 3. The three nights are consistent enough (3.0x to 4.2x a normal day) that shrinkage was discarding a real effect. v2's big-night multiplier is +170%; Halloween's is +16%.

For 2026 that means Thanksgiving Eve $5,024 (v1: $1,857), Dec 23 $4,843 (v1: $1,790), New Year's Eve $6,251 (v1: $2,311) and Halloween $7,742 (v1: $6,701).

Two cautions. λ was chosen on three nights, and all three come from a single year, so if 2025's big nights were unusual, v2 will overshoot. And the 80% ranges on those nights are very wide. The frozen v2 forecast is the test.

## 8c. Forecasting people instead of dollars

Kieran wants to know how busy Sawtooth will be, basically how many people are in town, so he can scale staff, ordering and prep. Toast records three daily volume measures: net sales, orders (checks) and guests. Sales carries noise unrelated to how many people came in: private-party deposits and invoices land on payment days, and spend per order fell about 20% in 2026.

Below: the same hybrid model and inputs with each of the three as the target, scored on the rolling backtest (refits every 3 months, as in section 8) and on the six-month backtest (section 2). Errors are in each target's own units, so read them as "how far off, in percent".

In [ ]:
SCORE_COLS = {"daily WAPE": "{:.1%}", "weekly WAPE": "{:.1%}", "typical week": "{:.1%}", "weeks >20% off": "{:.0%}",
              "worst week": "{:.0%}"}


def rolling_scores(target, skip_dates=None, fit_fn=None, predict_fn=None, **cols):
    # Rolling backtest with refits every 3 months; returns weekly scores and weekly log errors per horizon.
    # fit_fn and predict_fn default to the hybrid (section 8); section 11 passes the log-baseline version.
    fit_fn, predict_fn = fit_fn or hybrid_fit, predict_fn or hybrid_predict
    pieces = []
    for r in SCHEDULES["every 3 months"]:
        train, rows = od[od.date <= r], od[(od.date > r) & (od.date <= ROLL_END)]
        pred = predict_fn(fit_fn(train, target=target, **cols), rows)[0.5]
        pieces.append(pd.DataFrame({"refit": r, "date": rows.date, "actual": rows[target], "pred": pred}))
    rr = pd.concat(pieces, ignore_index=True)
    out = {}
    for horizon in (7, 28):
        monday = rr.date - pd.to_timedelta(rr.date.dt.day_of_week, unit="D")
        p = rr[rr.refit <= monday - pd.Timedelta(days=horizon)].sort_values("refit").groupby("date").tail(1)
        p = p[p.date.between(SCORE_START, SCORE_END)]
        if skip_dates is not None:
            p = p[~p.date.isin(skip_dates)]
        wk = p.groupby(p.date.dt.to_period("W-SUN"))[["actual", "pred"]].sum()
        pct = (wk.pred - wk.actual) / wk.actual
        out[horizon] = ({"daily WAPE": (p.pred - p.actual).abs().sum() / p.actual.sum(),
                         "weekly WAPE": (wk.pred - wk.actual).abs().sum() / wk.actual.sum(),
                         "typical week": pct.abs().median(), "weeks >20% off": (pct.abs() > 0.2).mean(),
                         "worst week": pct.abs().max()}, np.log(wk.actual / wk.pred))
    return out


TARGETS = {"sales ($)": "net_sales", "orders": "orders", "guests": "guests"}
target_runs = {label: rolling_scores(t) for label, t in TARGETS.items()}
pd.DataFrame([{"target": label, "days ahead": h, **target_runs[label][h][0]} for label in TARGETS for h in (7, 28)]
             ).style.format(SCORE_COLS).hide(axis="index")

In [ ]:
# Same comparison on the six-month backtest (section 2's folds, which also cover 2025)
rows = []
for label, t in TARGETS.items():
    for fold in FOLDS:
        train, test = split(fold)
        wk = test.assign(pred=hybrid_predict(hybrid_fit(train, target=t), test)[0.5]).groupby(
            test.date.dt.to_period("W-SUN"))[[t, "pred"]].sum()
        pct = (wk.pred - wk[t]) / wk[t]
        rows.append({"target": label, "fold": fold, "weekly WAPE": (wk.pred - wk[t]).abs().sum() / wk[t].sum(),
                     "weeks >20% off": (pct.abs() > 0.2).mean()})
pd.DataFrame(rows).pivot(index="target", columns="fold").loc[list(TARGETS)].style.format("{:.1%}")

**Features that targeted the worst weeks.** The worst sales weeks pointed at Commencement week (families in town, labeled as a break) and at spring break being lumped with December and September breaks. Both were added as features and tested; neither helped, for sales or for guests:

In [ ]:
EV_V1 = [f"ev_{k}" for k in BIG] + ["football_home", "commencement_week", "private_event", "show_comedy_improv",
                                    "show_music", "week_of_term", "weekday"]
TERM2 = ["weekday", "is_2026", "term2"]
TESTS = {"original inputs": {},
         "+ Commencement week, breaks split by season": {"base_cols": TERM2, "event_cols": EV_V1},
         "same, without private-event dates": {"base_cols": TERM2, "event_cols": [c for c in EV_V1 if c != "private_event"]}}
rows = []
for label, t in {"sales ($)": "net_sales", "guests": "guests"}.items():
    for name, cols in TESTS.items():
        m = (target_runs[label] if not cols else rolling_scores(t, **cols))[7][0]
        rows.append({"target": label, "inputs": name, **m})
pd.DataFrame(rows).style.format(SCORE_COLS).hide(axis="index")

**Can the guest counts be trusted?** Guest counts depend on staff entering them. Guests per order slid over 2026, which fits the shift from food (groups at tables) to drinks (individual tabs), but could also reflect a change in how counts are entered. Orders don't depend on staff entry and show the same gain over sales, so the improvement isn't an artifact of guest counts. Worth confirming with Kieran how guest counts are entered at the bar.

In [ ]:
q = od.groupby(od.date.dt.to_period("Q"))[["orders", "guests", "net_sales"]].sum()
display(pd.DataFrame({"guests per order": q.guests / q.orders, "sales per guest": q.net_sales / q.guests}).style.format(
    {"guests per order": "{:.2f}", "sales per guest": "${:.2f}"}))
print(f"Days with fewer guests than orders: {int((od.guests < od.orders).sum())}; "
      f"days with exactly one guest per order: {(od.guests == od.orders).mean():.0%}")

**Takeaway.** Forecasting guests instead of sales is the biggest improvement in this notebook, and it needs no new data. Same model and inputs, a week ahead:

| | Sales | Guests |
|---|---|---|
| Typical week off by | 11.3% | 9.4% |
| Weeks more than 20% off | 32% | 8% (3 of 37) |
| Weekly WAPE | 18.3% | 11.3% |
| Worst week | 72% | 42% |

It holds on the six-month backtest too: guests have lower weekly error than sales in all three periods (17.5%, 14.6%, 8.9% vs 22.5%, 22.1%, 13.7%). Orders, which don't depend on staff entering counts, also beat sales. The features aimed at the worst sales weeks (Commencement week, breaks split by season) didn't help either target, so they aren't used.

Several targets were compared on the same test periods, so some optimism applies, but two backtests and two head-count measures agree.

## 9. Frozen forecast: Oct 22 to Dec 31, 2026

Made on 2026-11-01 from data through 2026-10-21. It's saved to `forecasts/` and committed to git, and re-running this notebook never overwrites it (the training cutoff stays at Oct 21 even after newer exports are added). Oct 22 to Nov 1 had already happened when it was made, but those sales were unseen; Nov 2 onward was genuinely in the future.

**Model:** the hybrid (section 2d) refit on all data through Oct 21, with its 80% daily range. This is the primary forecast. The linear model and the weekday median are saved alongside it.

**Pre-registered alternative:** `hybrid_holiday_blend` averages the hybrid with last year's same day (scaled by that weekday's year-over-year level as of Oct 21) on the six holiday days listed below, where the model has no feature. Everywhere else it equals the hybrid. Scoring compares all four.

**Assumptions** (to confirm with Kieran):
- **Closed:** Sundays and Mondays (the pattern since May 25, 2026) unless a private event is booked that day, plus Thanksgiving (Nov 26), Dec 24 and Dec 25, which were all effectively closed in 2025.
- **Known bookings:** deposits already in Toast for Oct 29, Nov 12, Nov 16 (a Monday) and Nov 17.
- **No future shows known:** no ticket presales after Oct 21 appear in the data, so the weekly comedy nights aren't flagged.
- **Dartmouth:** fall classes end Nov 17, finals run Nov 20-25, then interim. Home football Oct 30 (Harvard) and Nov 21 (Brown).
- **No holiday feature:** the busiest holiday days of 2025 are listed below with last year's sales. They aren't adjusted.

In [ ]:
import hashlib

FORECAST_MADE, DATA_THROUGH = pd.Timestamp("2026-11-01"), pd.Timestamp("2026-10-21")
HOLIDAY_CLOSED = {pd.Timestamp("2026-11-26"): "Thanksgiving", pd.Timestamp("2026-12-24"): "Christmas Eve",
                  pd.Timestamp("2026-12-25"): "Christmas"}
RISK = {"2026-11-25": ("Wednesday before Thanksgiving", "2025-11-26"), "2026-11-27": ("Day after Thanksgiving", "2025-11-28"),
        "2026-11-28": ("Saturday after Thanksgiving", "2025-11-29"), "2026-12-23": ("Dec 23", "2025-12-23"),
        "2026-12-26": ("Dec 26", "2025-12-26"), "2026-12-31": ("New Year's Eve", "2025-12-31")}

future = cal[cal.date.between(DATA_THROUGH + pd.Timedelta(days=1), FORECAST_END)].copy()
future["closed_reason"] = np.select(
    [future.date.isin(list(HOLIDAY_CLOSED)), future.weekday.isin(["Sunday", "Monday"]) & (future.private_event == 0)],
    [future.date.map(HOLIDAY_CLOSED).fillna(""), future.weekday], "")
future["expected_open"] = future.closed_reason == ""
open_future = future[future.expected_open]

train = od[od.date <= DATA_THROUGH]   # fixed cutoff, even after newer exports are added
frame = pd.concat([od, open_future])
h = hybrid_predict(hybrid_fit(train, frame=frame), open_future)
lin_model, X_lin = fit_quantile(M2P, train, frame=frame)
flag_cols = [f"ev_{k}" for k in BIG] + ["football_home", "private_event", "show_night"]
forecast = future[["date", "weekday", "expected_open", "closed_reason", "term", "week_of_term"]].assign(
    events=future[flag_cols].apply(lambda r: ", ".join(c.removeprefix("ev_") for c in flag_cols if r[c] == 1), axis=1),
    hybrid_forecast=h[0.5], hybrid_lo80=h[0.1], hybrid_hi80=h[0.9],
    linear_forecast=pd.Series(lin_model.predict(X_lin.loc[open_future.index]), index=open_future.index),
    weekday_median_forecast=open_future.weekday.map(train.groupby("weekday").net_sales.median()),
    outcome_existed_when_made=future.date <= FORECAST_MADE)

# Pre-registered alternative (not the primary): on the holiday days the model has no feature for, average the hybrid
# with last year's same day scaled by that weekday's year-over-year level as of the data cutoff (section 2c).
last_year = pd.Series({pd.Timestamp(d): SALES.get(pd.Timestamp(ly), np.nan) * yoy_level(pd.Timestamp(d), lag=(pd.Timestamp(d) - DATA_THROUGH).days)
                       for d, (_, ly) in RISK.items()})
forecast["hybrid_holiday_blend"] = forecast.hybrid_forecast
on = forecast.date.isin(last_year.index)
forecast.loc[on, "hybrid_holiday_blend"] = (forecast.loc[on, "hybrid_forecast"] + forecast.loc[on, "date"].map(last_year)) / 2

# Weekly ranges from the rolling backtest's weekly errors (3-month refits, 28 days ahead)
wp = pick("every 3 months", 28, weekly=True)
wk_bt = wp.groupby(wp.date.dt.to_period("W-SUN"))[["actual", "hybrid"]].sum()
lo_q, hi_q = np.quantile(np.log(wk_bt.actual / wk_bt.hybrid), [0.1, 0.9])
weekly = forecast.groupby(forecast.date.dt.to_period("W-SUN")).agg(
    days_in_forecast=("date", "size"), open_days=("expected_open", "sum"), hybrid_forecast=("hybrid_forecast", "sum"),
    hybrid_holiday_blend=("hybrid_holiday_blend", "sum"))
weekly["hybrid_lo80"], weekly["hybrid_hi80"] = weekly.hybrid_forecast * np.exp(lo_q), weekly.hybrid_forecast * np.exp(hi_q)
weekly.index = weekly.index.astype(str)
weekly.index.name = "week"

OUT = Path("forecasts")
OUT.mkdir(exist_ok=True)
DAILY_FILE = OUT / "frozen_forecast_daily_2026-10-22_to_2026-12-31.csv"
WEEKLY_FILE = OUT / "frozen_forecast_weekly_2026-10-22_to_2026-12-31.csv"
if DAILY_FILE.exists():
    saved = pd.read_csv(DAILY_FILE, parse_dates=["date"]).set_index("date").hybrid_forecast
    drift = (saved - forecast.set_index("date").hybrid_forecast).abs().max()
    print(f"Frozen forecast already saved, not overwritten. Largest difference from re-running it now: ${drift:,.2f}")
else:
    forecast.to_csv(DAILY_FILE, index=False, float_format="%.2f")
    weekly.to_csv(WEEKLY_FILE, float_format="%.2f")
    print(f"Saved {DAILY_FILE} and {WEEKLY_FILE}")
for f in (DAILY_FILE, WEEKLY_FILE):
    print(f"sha256 {f.name}: {hashlib.sha256(f.read_bytes()).hexdigest()}")
weekly.style.format({c: "${:,.0f}" for c in ["hybrid_forecast", "hybrid_lo80", "hybrid_hi80", "hybrid_holiday_blend"]})

In [ ]:
f = forecast.set_index("date")
pd.DataFrame([{"date": d, "day": label, "weekday": f.loc[d, "weekday"], "hybrid forecast": f.loc[d, "hybrid_forecast"],
               "80% range": f"${f.loc[d, 'hybrid_lo80']:,.0f} to ${f.loc[d, 'hybrid_hi80']:,.0f}",
               "same day in 2025": SALES.get(pd.Timestamp(ly), np.nan),
               "holiday-blend alternative": f.loc[d, "hybrid_holiday_blend"]} for d, (label, ly) in RISK.items()]).style.format(
    {"hybrid forecast": "${:,.0f}", "same day in 2025": "${:,.0f}", "holiday-blend alternative": "${:,.0f}"}).hide(axis="index")

### 9b. Frozen forecast v2 (with holidays)

Registered the same day as v1 (Nov 1, 2026), after v1 and before any of these days' sales were seen. Same training data, same model, times the holiday multipliers from section 8b. Closures come from the holiday rules (`hol_closed`) plus the Sunday/Monday pattern, which gives the same closed days as v1 in this window. Saved as separate files; v1 isn't touched.

In [ ]:
assert set(future.loc[future.hol_closed == 1, "date"]) <= set(future.loc[~future.expected_open, "date"])
fit_v2 = hybrid_fit(train, frame=frame)
q = hybrid_predict(fit_v2, open_future)
mult = holiday_multiplier(open_future, holiday_effects(train, fit_v2, HOLIDAY_LAMBDAS))
holiday_label = np.select([future.hol_big_night == 1, future.hol_halloween == 1, future.hol_closed == 1],
                          ["big night", "Halloween", "closed holiday"], "")
forecast_v2 = future[["date", "weekday", "expected_open", "closed_reason"]].assign(
    holiday=holiday_label, v2_forecast=q[0.5] * mult, v2_lo80=q[0.1] * mult, v2_hi80=q[0.9] * mult,
    v1_hybrid_forecast=q[0.5], outcome_existed_when_made=future.date <= FORECAST_MADE)
weekly_v2 = forecast_v2.groupby(forecast_v2.date.dt.to_period("W-SUN")).agg(
    days_in_forecast=("date", "size"), open_days=("expected_open", "sum"), v2_forecast=("v2_forecast", "sum"))
weekly_v2["v2_lo80"], weekly_v2["v2_hi80"] = weekly_v2.v2_forecast * np.exp(lo_q), weekly_v2.v2_forecast * np.exp(hi_q)
weekly_v2.index = weekly_v2.index.astype(str)
weekly_v2.index.name = "week"

DAILY_V2 = OUT / "frozen_forecast_v2_daily_2026-10-22_to_2026-12-31.csv"
WEEKLY_V2 = OUT / "frozen_forecast_v2_weekly_2026-10-22_to_2026-12-31.csv"
if DAILY_V2.exists():
    saved_v2 = pd.read_csv(DAILY_V2, parse_dates=["date"]).set_index("date").v2_forecast
    print(f"Frozen v2 forecast already saved, not overwritten. Largest difference from re-running it now: "
          f"${(saved_v2 - forecast_v2.set_index('date').v2_forecast).abs().max():,.2f}")
else:
    forecast_v2.to_csv(DAILY_V2, index=False, float_format="%.2f")
    weekly_v2.to_csv(WEEKLY_V2, float_format="%.2f")
    print(f"Saved {DAILY_V2} and {WEEKLY_V2}")
for f in (DAILY_V2, WEEKLY_V2):
    print(f"sha256 {f.name}: {hashlib.sha256(f.read_bytes()).hexdigest()}")
display(forecast_v2[forecast_v2.holiday.isin(["big night", "Halloween"])].style.format(
    {"date": "{:%Y-%m-%d}", "v2_forecast": "${:,.0f}", "v2_lo80": "${:,.0f}", "v2_hi80": "${:,.0f}",
     "v1_hybrid_forecast": "${:,.0f}"}).hide(axis="index"))
weekly_v2.style.format({c: "${:,.0f}" for c in ["v2_forecast", "v2_lo80", "v2_hi80"]})

### 9c. Frozen forecast v3: guests

Registered Nov 1, 2026, after v1 and v2 and before any of these days' outcomes were seen. Same model and inputs as v1 (section 2d), but forecasting guest counts (section 8c), times holiday multipliers re-estimated for guests with the same leave-one-out choice of shrinkage as section 8b. Weekly 80% ranges come from the rolling backtest's weekly guest errors (refits every 3 months, 28 days ahead). Same closed days as v1 and v2.

In [ ]:
def choose_big_night_lambda(hist, target, **cols):
    # Leave-one-out over past big nights: total absolute error for each candidate lambda.
    errs = dict.fromkeys(LAMBDAS, 0.0)
    for idx in hist.index[hist.hol_big_night == 1]:
        tr, one = hist.drop(index=idx), hist.loc[[idx]]
        fit = hybrid_fit(tr, target=target, **cols)
        base = hybrid_predict(fit, one)[0.5].iloc[0]
        for lam in LAMBDAS:
            eff_one = holiday_effects(tr, fit, {"hol_big_night": lam}, target=target)
            errs[lam] += abs(one[target].iloc[0] - base * holiday_multiplier(one, eff_one).iloc[0])
    return min(errs, key=errs.get), errs


lam_g, lam_errs = choose_big_night_lambda(od[od.date <= DATA_THROUGH], "guests")
print("Leave-one-out guest error on the 3 big nights by lambda:", {k: round(v) for k, v in lam_errs.items()}, "-> chosen", lam_g)
fit_v3 = hybrid_fit(train, frame=frame, target="guests")
eff_v3 = holiday_effects(train, fit_v3, {"hol_big_night": lam_g, "hol_halloween": 3}, target="guests")
print("Guest holiday effects:", {k: f"{np.expm1(v):+.0%}" for k, v in eff_v3.items()})
q3 = hybrid_predict(fit_v3, open_future)
mult3 = holiday_multiplier(open_future, eff_v3)
forecast_v3 = future[["date", "weekday", "expected_open", "closed_reason"]].assign(
    holiday=holiday_label, guests_forecast=q3[0.5] * mult3, guests_lo80=q3[0.1] * mult3, guests_hi80=q3[0.9] * mult3,
    outcome_existed_when_made=future.date <= FORECAST_MADE)
g_lo, g_hi = np.quantile(target_runs["guests"][28][1], [0.1, 0.9])
weekly_v3 = forecast_v3.groupby(forecast_v3.date.dt.to_period("W-SUN")).agg(
    days_in_forecast=("date", "size"), open_days=("expected_open", "sum"), guests_forecast=("guests_forecast", "sum"))
weekly_v3["guests_lo80"], weekly_v3["guests_hi80"] = weekly_v3.guests_forecast * np.exp(g_lo), weekly_v3.guests_forecast * np.exp(g_hi)
weekly_v3.index = weekly_v3.index.astype(str)
weekly_v3.index.name = "week"

DAILY_V3 = OUT / "frozen_forecast_v3_guests_daily_2026-10-22_to_2026-12-31.csv"
WEEKLY_V3 = OUT / "frozen_forecast_v3_guests_weekly_2026-10-22_to_2026-12-31.csv"
if DAILY_V3.exists():
    saved_v3 = pd.read_csv(DAILY_V3, parse_dates=["date"]).set_index("date").guests_forecast
    print(f"Frozen v3 forecast already saved, not overwritten. Largest difference from re-running it now: "
          f"{(saved_v3 - forecast_v3.set_index('date').guests_forecast).abs().max():.2f} guests")
else:
    forecast_v3.to_csv(DAILY_V3, index=False, float_format="%.2f")
    weekly_v3.to_csv(WEEKLY_V3, float_format="%.2f")
    print(f"Saved {DAILY_V3} and {WEEKLY_V3}")
for f in (DAILY_V3, WEEKLY_V3):
    print(f"sha256 {f.name}: {hashlib.sha256(f.read_bytes()).hexdigest()}")
weekly_v3.style.format({c: "{:,.0f}" for c in ["guests_forecast", "guests_lo80", "guests_hi80"]})

### 9d. Product rule: booked private events

A private buyout makes a busy night look empty in Toast: the guests are there, but they're billed on an invoice, not rung up as walk-ins. Kieran knows these nights in advance from his bookings, so the forecast shouldn't try to predict walk-ins on them.

**Rule** (uses only what's known when the booking is made; Kieran's calendar overrides it once we have it). A booked event is treated as a **full buyout** if any of these hold:
1. It's on a day Sawtooth is normally closed (Sunday, or a Monday from May 25, 2026), so the bar opens only for the event.
2. It's a reunion.
3. It's a formal on a Friday or Saturday.

On full-buyout nights the forecast shows **booked: private event** instead of a walk-in number. Every other booked night keeps the v3 forecast, with a note. The rule was written after looking at how past bookings behaved, so how well it fits history is optimistic; it was registered on Nov 1, 2026, before November's outcomes, and that's its real test.

Bookings come from Toast itself until Kieran's calendar is available: dated Event-category item names (deposits, remainders, formals) plus invoice numbers, which encode the event date as month, day, two-digit year and a two-digit sequence (for example "Tuck Fall Reunion 10/4 Invoice #10042502"). Off-site catering is left out. Customer names are never shown.

In [ ]:
def invoice_event_dates():
    # Event dates encoded in Toast invoice numbers: M(M)DDYY + 2-digit sequence, e.g. 7262501 -> Jul 26, 2025.
    rows = []
    for p in sorted(DATA.glob("ProductMix_202[56]-*.xlsx")):
        start, end = (pd.Timestamp(x) for x in p.stem.split("_")[1:3])
        mid = start + (end - start) / 2
        oi = pd.read_excel(p, sheet_name="Open items")
        for name in oi.loc[oi["Sales Category"] == "Event", "Open item"].astype(str):
            m = re.search(r"Invoice\s*#?\s*(\d{7,8})", name)
            if not m:
                continue
            mmdd, yy = m[1][:-4], m[1][-4:-2]
            candidates = []
            for k in (1, 2):   # the month is one or two digits
                try:
                    candidates.append(pd.Timestamp(2000 + int(yy), int(mmdd[:k]), int(mmdd[k:])))
                except ValueError:
                    pass
            if candidates:
                rows.append({"date": min(candidates, key=lambda c: abs(c - mid)), "name": name})
    return pd.DataFrame(rows)


def classify_bookings(bookings):
    # One row per booked date: kind of event and whether the rule treats it as a full buyout.
    names = bookings.groupby("date").name.apply(lambda x: " ".join(x).lower())
    kind = np.select([names.str.contains("reunion"), names.str.contains("formal"), names.str.contains("grad")],
                     ["reunion", "formal", "grad party"], "party")
    out = pd.DataFrame({"date": names.index, "kind": kind})
    weekday = out.date.dt.day_name()
    normally_closed = (weekday == "Sunday") | ((weekday == "Monday") & (out.date >= "2026-05-25"))
    weekend_formal = (out.kind == "formal") & weekday.isin(["Friday", "Saturday"])
    out["buyout"] = normally_closed | (out.kind == "reunion") | weekend_formal
    out["reason"] = np.select([normally_closed, out.kind == "reunion", weekend_formal],
                              ["opens only for the event", "reunion", "weekend formal"], "")
    return out


named = dated_item_names("Open items", lambda oi: oi.loc[oi["Sales Category"] == "Event", "Open item"])
bookings = pd.concat([named, invoice_event_dates()], ignore_index=True)
bookings = bookings[~bookings.name.str.contains("catering", case=False)]
booked = classify_bookings(bookings)

# How the rule classifies past bookings, against what actually happened (walk-in guests vs a normal same weekday)
normal = od[~od.date.isin(booked.date)].groupby("weekday").guests.median()
hist = booked[booked.date <= DATA_THROUGH].merge(od[["date", "weekday", "guests"]], on="date", how="left")
hist["walk-in vs normal"] = hist.guests / hist.weekday.map(normal)
display(hist.groupby(["buyout", "kind"]).agg(days=("date", "size"), median_walk_in=("walk-in vs normal", "median"),
                                             below_0_6=("walk-in vs normal", lambda r: int((r < 0.6).sum()))).style.format(
    {"median_walk_in": "{:.2f}x"}, na_rep="no sales row"))
low = hist["walk-in vs normal"] < 0.6
print(f"Rule flags {int(hist.buyout.sum())} of {len(hist)} past booked days as buyouts. Of the {int(low.sum())} booked days where "
      f"walk-ins fell below 0.6x normal, the rule catches {int((low & hist.buyout).sum())}; of the days it flags, "
      f"{int((hist.buyout & ~low & hist.guests.notna()).sum())} had normal walk-in traffic.")

In [ ]:
# Backtest effect: v3 (guests) scored on every open day vs on walk-in nights only (no walk-in forecast on rule buyouts)
buyout_dates = set(booked.loc[booked.buyout, "date"])
with_rule = rolling_scores("guests", skip_dates=buyout_dates)
pd.DataFrame([{"scored on": "every open day", "days ahead": h, **target_runs["guests"][h][0]} for h in (7, 28)]
             + [{"scored on": "walk-in nights (rule buyouts get no walk-in forecast)", "days ahead": h, **with_rule[h][0]} for h in (7, 28)]
             ).style.format(SCORE_COLS).hide(axis="index")

In [ ]:
# What Kieran would see for Oct 22 to Dec 31, 2026: the frozen v3 forecast with closures and the booking rule applied
v3 = pd.read_csv(DAILY_V3, parse_dates=["date"]).merge(booked, on="date", how="left")
v3["buyout"] = v3.buyout.astype("boolean").fillna(False)
v3["status"] = np.select([v3.buyout, ~v3.expected_open, v3.kind.notna()],
                         ["booked: private event", "closed: " + v3.closed_reason, "open, with a booked " + v3.kind.fillna("")], "open")
walk_in = v3.status.str.startswith("open")
presentation = v3.assign(
    walk_in_guests=v3.guests_forecast.where(walk_in).round(0), low_80=v3.guests_lo80.where(walk_in).round(0),
    high_80=v3.guests_hi80.where(walk_in).round(0))[["date", "weekday", "status", "holiday", "walk_in_guests", "low_80", "high_80"]]
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
display(presentation[presentation.status != "open"].style.format(
    {"date": "{:%a %b %d}", "walk_in_guests": "{:.0f}", "low_80": "{:.0f}", "high_80": "{:.0f}"}, na_rep="").hide(axis="index"))
week = presentation.groupby(presentation.date.dt.to_period("W-SUN")).agg(
    walk_in_guests=("walk_in_guests", "sum"), booked_private_events=("status", lambda x: int((x == "booked: private event").sum())))
week.index = week.index.astype(str)
week.style.format({"walk_in_guests": "{:,.0f}"})

### 9e. Widening the 80% ranges

Backtests showed the guest ranges were too narrow: about 68% of days landed inside an "80%" range. The fix stretches each range around the forecast by a factor k (on a log scale, so ranges stay proportional), without cheating:

1. Pick k on Feb-Jun 2026 forecasts from the rolling backtest (refits every 3 months, forecasts made at least 28 days ahead): the smallest k that puts 80% of actual days inside.
2. Check that k on Jul-Oct 2026, which played no part in choosing it, and report the coverage as is.
3. For the live calendar, recompute k on all backtest days. Weekly ranges get the same treatment, using each week's total.

In [ ]:
def rolling_quantiles(target="guests", horizon=28):
    # Rolling backtest (refits every 3 months) keeping the 10th, 50th and 90th percentile forecasts.
    pieces = []
    for r in SCHEDULES["every 3 months"]:
        train, rows = od[od.date <= r], od[(od.date > r) & (od.date <= ROLL_END)]
        q = hybrid_predict(hybrid_fit(train, target=target), rows)
        pieces.append(pd.DataFrame({"refit": r, "date": rows.date, "actual": rows[target], "lo": q[0.1], "mid": q[0.5], "hi": q[0.9]}))
    rr = pd.concat(pieces, ignore_index=True)
    monday = rr.date - pd.to_timedelta(rr.date.dt.day_of_week, unit="D")
    p = rr[rr.refit <= monday - pd.Timedelta(days=horizon)].sort_values("refit").groupby("date").tail(1)
    return p[p.date.between(SCORE_START, SCORE_END)].reset_index(drop=True)


def widen(mid, lo, hi, k):
    return mid * np.exp(k * np.log(lo.clip(lower=0.5) / mid)), mid * np.exp(k * np.log(hi / mid))


def daily_coverage(df, k):
    lo, hi = widen(df.mid, df.lo, df.hi, k)
    return df.actual.between(lo, hi).mean()


qd = rolling_quantiles()
DEV_END = pd.Timestamp("2026-06-28")
dev_d, conf_d = qd[qd.date <= DEV_END], qd[qd.date > DEV_END]
KS = np.round(np.arange(1.0, 3.01, 0.05), 2)
k_dev = next(k for k in KS if daily_coverage(dev_d, k) >= 0.80)
K_DAILY = next(k for k in KS if daily_coverage(qd, k) >= 0.80)

wk = qd.groupby(qd.date - pd.to_timedelta(qd.date.dt.day_of_week, unit="D"))[["actual", "mid"]].sum()
log_err = np.log(wk.actual / wk.mid)
dev_w, conf_w = log_err[log_err.index <= DEV_END], log_err[log_err.index > DEV_END]
w_dev = np.quantile(dev_w, [0.1, 0.9])
W_LO, W_HI = np.quantile(log_err, [0.1, 0.9])

display(pd.DataFrame([
    {"range": "daily, as frozen (k = 1)", "Feb-Jun coverage": daily_coverage(dev_d, 1.0), "Jul-Oct coverage (held out)": daily_coverage(conf_d, 1.0)},
    {"range": f"daily, widened (k = {k_dev}, chosen on Feb-Jun)", "Feb-Jun coverage": daily_coverage(dev_d, k_dev),
     "Jul-Oct coverage (held out)": daily_coverage(conf_d, k_dev)},
    {"range": f"weekly ({np.expm1(w_dev[0]):+.0%} to {np.expm1(w_dev[1]):+.0%}, chosen on Feb-Jun)",
     "Feb-Jun coverage": dev_w.between(*w_dev).mean(), "Jul-Oct coverage (held out)": conf_w.between(*w_dev).mean()},
]).style.format({"Feb-Jun coverage": "{:.0%}", "Jul-Oct coverage (held out)": "{:.0%}"}).hide(axis="index"))
print(f"For the live calendar (all backtest days and weeks): daily k = {K_DAILY}; weekly range {np.expm1(W_LO):+.0%} to "
      f"{np.expm1(W_HI):+.0%} around the forecast. {len(dev_d)} Feb-Jun days, {len(conf_d)} Jul-Oct days; "
      f"{len(dev_w)} and {len(conf_w)} weeks.")
# How wide would nightly ranges need to be to hold up in Jul-Oct? (shown to explain the decision, not used to pick k)
display(pd.DataFrame([{"k": k, "Jul-Oct nightly coverage": daily_coverage(conf_d, k),
                       "typical nightly range": "{:.0%} to {:.0%} of the forecast".format(
                           *(np.nanmedian(b / conf_d.mid) for b in widen(conf_d.mid, conf_d.lo, conf_d.hi, k)))}
                      for k in (1.0, 1.4, 2.0, 2.5)]).style.format({"Jul-Oct nightly coverage": "{:.0%}"}).hide(axis="index"))

**Takeaway.** Weekly ranges hold up: chosen on Feb-Jun, they contained 75% of the held-out Jul-Oct weeks (12 of 16), close to the 80% target. Nightly ranges don't: widening chosen on Feb-Jun covered only 56% of held-out Jul-Oct nights, and ranges that did hold up in Jul-Oct would run from about half to 1.5 times the forecast, too wide to plan a night around. Single nights swung more in Jul-Oct 2026, plausibly because lunch service ended and Mondays closed. So the calendar gives each week a range, and gives each night its expected count and how busy that is against a normal night, without a range.

### 9f. Model v5: weekday levels since lunch ended

Since lunch service ended (assumed July 1, 2026: lunch sales were normal in the Jan-Jun export and nearly gone in Jul-Oct; to confirm with Kieran), v3 has run high on Wednesdays and Thursdays and low on Saturdays. v3 can't represent that, because its 2026 adjustment shifts every weekday by the same amount. v5 gives each weekday its own level after July 1, 2026; everything else is the same as v3.

The usual backtest can't check this, because the Feb-Jun weeks have no data from after lunch ended. Instead, both models are trained on data through Aug 31, 2026 and scored on Sep 1 to Oct 21, which neither saw. That check was designed after seeing the Jul-Oct bias, so it isn't pristine; the real test is the frozen v5 forecast below.

**Decision rule, set before running:** the calendar uses v5 if, on Sep 1-Oct 21, its weekly WAPE is lower than v3's and its average weekday bias on Wednesday, Thursday and Saturday is smaller. v5 is frozen either way.

In [ ]:
BASE_V5 = ["weekday_regime", "is_2026", "term"]
train_t, test_t = od[od.date <= "2026-08-31"], od[od.date.between("2026-09-01", "2026-10-21")]
check = {}
for name, cols in {"v3": {}, "v5": {"base_cols": BASE_V5}}.items():
    d = test_t.assign(pred=hybrid_predict(hybrid_fit(train_t, target="guests", **cols), test_t)[0.5])
    wk = d.groupby(d.date - pd.to_timedelta(d.date.dt.day_of_week, unit="D"))[["guests", "pred"]].sum()
    pct = wk.pred / wk.guests - 1
    by_wd = d.groupby("weekday")[["pred", "guests"]].sum()
    bias = (by_wd.pred / by_wd.guests - 1).reindex(["Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"])
    check[name] = {"daily WAPE": (d.pred - d.guests).abs().sum() / d.guests.sum(),
                   "weekly WAPE": (wk.pred - wk.guests).abs().sum() / wk.guests.sum(), "typical week": pct.abs().median(),
                   "weeks >20% off": f"{int((pct.abs() > 0.2).sum())} of {len(wk)}", **{f"{w[:3]} bias": b for w, b in bias.items()}}
check = pd.DataFrame(check).T
display(check.style.format({c: "{:+.0%}" for c in check.columns if "bias" in c}
                           | {"daily WAPE": "{:.1%}", "weekly WAPE": "{:.1%}", "typical week": "{:.1%}"}))
avg_bias = {m: check.loc[m, ["Wed bias", "Thu bias", "Sat bias"]].astype(float).abs().mean() for m in check.index}
USE_V5 = bool(check.loc["v5", "weekly WAPE"] < check.loc["v3", "weekly WAPE"] and avg_bias["v5"] < avg_bias["v3"])
print(f"Average |bias| on Wed/Thu/Sat: v3 {avg_bias['v3']:.0%}, v5 {avg_bias['v5']:.0%}. "
      f"Weekly WAPE: v3 {check.loc['v3', 'weekly WAPE']:.1%}, v5 {check.loc['v5', 'weekly WAPE']:.1%}. Calendar uses {'v5' if USE_V5 else 'v3'}.")

**Frozen v5 forecast.** Registered Nov 1, 2026, before any of these days' outcomes were seen; v1, v2 and v3 are unchanged. Trained on all data through Oct 21, with holiday multipliers re-estimated for v5 (same leave-one-out choice as 9c), the same closed days and booking rule, and the same weekly ranges as the calendar (9e).

In [ ]:
lam5, lam5_errs = choose_big_night_lambda(od[od.date <= DATA_THROUGH], "guests", base_cols=BASE_V5)
fit_v5 = hybrid_fit(train, frame=frame, target="guests", base_cols=BASE_V5)
eff_v5 = holiday_effects(train, fit_v5, {"hol_big_night": lam5, "hol_halloween": 3}, target="guests")
print("v5 big-night lambda:", lam5, "| holiday effects:", {k: f"{np.expm1(v):+.0%}" for k, v in eff_v5.items()})
q5 = hybrid_predict(fit_v5, open_future)
mult5 = holiday_multiplier(open_future, eff_v5)
forecast_v5 = future[["date", "weekday", "expected_open", "closed_reason"]].assign(
    holiday=holiday_label, guests_forecast=q5[0.5] * mult5, guests_lo80=q5[0.1] * mult5, guests_hi80=q5[0.9] * mult5,
    outcome_existed_when_made=future.date <= FORECAST_MADE)
weekly_v5 = forecast_v5.groupby(forecast_v5.date.dt.to_period("W-SUN")).agg(
    days_in_forecast=("date", "size"), open_days=("expected_open", "sum"), guests_forecast=("guests_forecast", "sum"))
weekly_v5["guests_lo80"], weekly_v5["guests_hi80"] = weekly_v5.guests_forecast * np.exp(W_LO), weekly_v5.guests_forecast * np.exp(W_HI)
weekly_v5.index = weekly_v5.index.astype(str)
weekly_v5.index.name = "week"

DAILY_V5 = OUT / "frozen_forecast_v5_guests_daily_2026-10-22_to_2026-12-31.csv"
WEEKLY_V5 = OUT / "frozen_forecast_v5_guests_weekly_2026-10-22_to_2026-12-31.csv"
if DAILY_V5.exists():
    saved_v5 = pd.read_csv(DAILY_V5, parse_dates=["date"]).set_index("date").guests_forecast
    print(f"Frozen v5 forecast already saved, not overwritten. Largest difference from re-running it now: "
          f"{(saved_v5 - forecast_v5.set_index('date').guests_forecast).abs().max():.2f} guests")
else:
    forecast_v5.to_csv(DAILY_V5, index=False, float_format="%.2f")
    weekly_v5.to_csv(WEEKLY_V5, float_format="%.2f")
    print(f"Saved {DAILY_V5} and {WEEKLY_V5}")
for f in (DAILY_V5, WEEKLY_V5):
    print(f"sha256 {f.name}: {hashlib.sha256(f.read_bytes()).hexdigest()}")
CAL_SOURCE = DAILY_V5 if USE_V5 else DAILY_V3

v3_daily = pd.read_csv(DAILY_V3, parse_dates=["date"]).set_index("date").guests_forecast
cmp = forecast_v5.set_index("date").assign(v3=v3_daily)
cmp = cmp[cmp.guests_forecast.notna()]
cmp.groupby("weekday")[["v3", "guests_forecast"]].mean().rename(columns={"guests_forecast": "v5"}).reindex(
    ["Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Monday"]).dropna().round(0)

### 9g. The calendar for Kieran

The product: a week-by-week calendar of expected guests at Sawtooth for Oct 22 to Dec 31, 2026. Each open day shows expected guests, how that compares with a normal night of that weekday this fall, the widened range, and the reasons behind it (holidays, Dartmouth events, home games, bookings). Each week shows its total, a weekly range, and where Dartmouth is in its term. Reasons let Kieran check the forecast against what he already knows.

Numbers come from the frozen forecast picked in 9f (v5 if it passed its check, otherwise v3; both unchanged), with the booking rule (9d) and the weekly ranges (9e). Oct 22 to Nov 1 had already happened when the forecast was made on Nov 1, but those sales were unseen.

In [ ]:
import html as _html

cal_idx = cal.set_index("date")


def term_context(d):
    for r in dc[dc.kind == "finals"].itertuples():
        if r.start_date <= d <= r.end_date:
            return f"{r.event.split()[0]} finals"
    for r in dc[dc.kind == "term_classes"].itertuples():
        if r.start_date <= d <= r.end_date:
            return f"{r.event} classes, week {(d - r.start_date).days // 7 + 1}"
        fin = dc[(dc.kind == "finals") & (dc.event == r.event)]
        if len(fin) and r.end_date < d < fin.start_date.min():
            return "Reading period before finals"
    return "Break: most students away"


def day_reasons(d):
    c, out = cal_idx.loc[d], []
    if c.hol_big_night:
        name = "Thanksgiving Eve" if d.month == 11 else ("New Year's Eve" if d.day == 31 else "Last night before Christmas")
        out.append(name + ": a top night of the year (about 3-4x a normal night in 2025)")
    if c.hol_halloween:
        out.append("Halloween")
    for r in dc[dc.kind.isin(BIG + ["football_home"]) & (dc.start_date <= d) & (dc.end_date >= d)].itertuples():
        out.append(re.sub(r"\s*\d{4}$", "", r.event).replace("Football vs", "Home football vs"))
    b = booked[booked.date == d]
    if len(b):
        out.append("Booked private event: the bar is reserved" if b.buyout.iloc[0] else f"Booked {b.kind.iloc[0]}; walk-in forecast shown")
    return out


def vs_normal(guests, weekday):
    ratio = guests / NORMAL[weekday]
    if ratio >= 1.5:
        return f"about {ratio:.1f}x a normal {weekday}"
    if abs(ratio - 1) < 0.15:
        return "about normal"
    return f"{abs(ratio - 1):.0%} {'above' if ratio > 1 else 'below'} normal"


v3c = pd.read_csv(CAL_SOURCE, parse_dates=["date"]).merge(booked[["date", "buyout"]], on="date", how="left")
v3c["buyout"] = v3c.buyout.astype("boolean").fillna(False)
# "Normal" = the forecast's own typical night of that weekday during fall classes (open, no event, booking or holiday),
# so the shading reflects who's in town rather than the model's weekday bias (section 9e takeaway)
fall_classes = dc[(dc.kind == "term_classes") & (dc.event == "Fall 2026")].iloc[0]
plain = v3c[v3c.expected_open & ~v3c.buyout & v3c.date.between(fall_classes.start_date, fall_classes.end_date)
            & (v3c.date.map(lambda d: len(day_reasons(d))) == 0)]
NORMAL = plain.groupby("weekday").guests_forecast.median()
print("Typical fall-class night in this forecast (median guests):", NORMAL.round(0).to_dict())
rows = []
for r in v3c.itertuples():
    status = "booked" if r.buyout else ("closed" if not r.expected_open else "open")
    is_open = status == "open"
    rows.append({"date": r.date, "weekday": r.weekday, "status": status,
                 "closed_reason": r.closed_reason if status == "closed" else "",
                 "expected_guests": round(r.guests_forecast) if is_open else None,
                 "vs_normal": vs_normal(r.guests_forecast, r.weekday) if is_open else "",
                 "reasons": "; ".join(day_reasons(r.date)), "term": term_context(r.date)})
calendar_df = pd.DataFrame(rows)
calendar_df.to_csv(OUTPUTS / "sawtooth_calendar_2026-10-22_to_2026-12-31.csv", index=False)
calendar_df[calendar_df.reasons != ""][["date", "weekday", "status", "expected_guests", "vs_normal", "reasons"]].style.format(
    {"date": "{:%a %b %d}", "expected_guests": "{:.0f}"}, na_rep="").hide(axis="index")

In [ ]:
LEVEL_COLORS = {"much busier": "#f6c4ad", "busier": "#fbe3d8", "about normal": "#fcfcfb", "quieter": "#e6effb", "much quieter": "#cfe0f6"}
CSS = ("body { background:#f9f9f7; color:#0b0b0b; font:14px/1.4 -apple-system, system-ui, sans-serif; margin:16px; }"
       "h1 { font-size:20px; margin:0 0 4px; } p.sub { color:#52514e; margin:0 0 12px; max-width:900px; }"
       ".wrap { overflow-x:auto; } table { border-collapse:separate; border-spacing:4px; min-width:900px; }"
       "th { text-align:left; font-weight:600; padding:12px 4px 2px; }"
       "td { vertical-align:top; width:14%; padding:8px; border:1px solid #e1e0d9; border-radius:6px; background:#fcfcfb; }"
       "td.closed { background:#efeeea; color:#898781; } td.booked { background:#ece9fb; } td.out { border:none; background:transparent; }"
       ".d { font-weight:600; font-size:12px; color:#52514e; } .g { font-size:18px; font-weight:700; margin-top:2px; }"
       ".r, .n { font-size:12px; color:#52514e; } .s { font-weight:600; margin-top:4px; } .why { font-size:12px; margin-top:4px; }"
       ".legend span { display:inline-block; padding:2px 8px; margin:0 6px 6px 0; border:1px solid #e1e0d9; border-radius:4px; font-size:12px; }")


def level_of(text):
    if "x a normal" in text:
        return "much busier"
    if text == "about normal":
        return "about normal"
    pct = int(text.split("%")[0])
    return "busier" if "above" in text else ("much quieter" if pct >= 40 else "quieter")


def cell(r):
    esc, head = _html.escape, f'<div class="d">{r.date:%a %b %-d}</div>'
    if r.status == "closed":
        return f'<td class="closed">{head}<div class="s">Closed: {esc(r.closed_reason)}</div></td>'
    if r.status == "booked":
        return f'<td class="booked">{head}<div class="s">Booked private event</div><div class="why">The bar is reserved; no walk-in forecast</div></td>'
    why = "".join(f'<div class="why">{esc(x)}</div>' for x in r.reasons.split("; ") if x)
    return (f'<td style="background:{LEVEL_COLORS[level_of(r.vs_normal)]}">{head}<div class="g">{r.expected_guests:.0f} guests</div>'
            f'<div class="n">{esc(r.vs_normal)}</div>{why}</td>')


day_rows = {r.date: r for r in calendar_df.itertuples()}
first = calendar_df.date.min() - pd.Timedelta(days=calendar_df.date.min().day_of_week)
week_html = []
for w in pd.date_range(first, calendar_df.date.max(), freq="7D"):
    days = pd.date_range(w, periods=7)
    total = sum(day_rows[d].expected_guests for d in days if d in day_rows and day_rows[d].status == "open")
    partial = " (partial week)" if sum(d in day_rows for d in days) < 7 else ""
    week_html.append(f'<tr><th colspan="7">Week of {w:%b %-d}{partial}: about {total:,.0f} guests (likely {total * np.exp(W_LO):,.0f} '
                     f'to {total * np.exp(W_HI):,.0f}) &middot; {_html.escape(term_context(w + pd.Timedelta(days=2)))}</th></tr>')
    week_html.append("<tr>" + "".join(cell(day_rows[d]) if d in day_rows else '<td class="out"></td>' for d in days) + "</tr>")

MODEL_NOTE = ("Weekday levels are re-estimated for the period since lunch service ended (assumed July 1, 2026; model v5). " if USE_V5 else
              "Known issue: since lunch service ended this summer, the model has run high on Wednesdays and Thursdays (about 25-30%) "
              "and low on Saturdays (about 15%) in backtests. ")
legend = "".join(f'<span style="background:{c}">{k}</span>' for k, c in LEVEL_COLORS.items())
legend += '<span style="background:#ece9fb">booked private event</span><span style="background:#efeeea">closed</span>'
page = ('<!doctype html><html lang="en"><head><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width, initial-scale=1"><title>Sawtooth Guest Calendar</title>'
        f'<style>{CSS}</style></head><body><h1>Sawtooth: expected guests, Oct 22 to Dec 31, 2026</h1>'
        '<p class="sub">Forecast made Nov 1, 2026 from Toast data through Oct 21. Each week shows a likely range for its total guests; '
        'in tests on past weeks, 3 to 4 weeks in 5 landed inside. Single nights swing much more (often half to 1.5 times the forecast), '
        'so each night shows how busy it should be compared with a typical night of that weekday during fall classes. ' + MODEL_NOTE + 'Oct 22 to Nov 1 had already happened when the forecast was made.</p>'
        f'<div class="legend">{legend}</div><div class="wrap"><table>{"".join(week_html)}</table></div></body></html>')
(OUTPUTS / "sawtooth_calendar_2026-10-22_to_2026-12-31.html").write_text(page)
print(f"Wrote {OUTPUTS / 'sawtooth_calendar_2026-10-22_to_2026-12-31.html'} and the matching CSV; normal nights this fall:",
      NORMAL.round(0).to_dict())

### 9h. Check: nights above any past night

A forecast above the most guests that weekday has had in the training data (through Oct 21, 2026) isn't necessarily wrong, since big nights are meant to be big, but it's the first place to look for a mistake. Flagged nights are listed for a person to review; no forecast is changed. Add each new frozen guest forecast to the list.

In [ ]:
most_by_weekday = od.groupby("weekday").guests.max()
above_rows = []
for name, path in {"v3": DAILY_V3, "v5": DAILY_V5}.items():
    f = pd.read_csv(path, parse_dates=["date"])
    f = f[f.expected_open & f.guests_forecast.notna()].assign(model=name, most_before=lambda d: d.weekday.map(most_by_weekday))
    above_rows.append(f[f.guests_forecast > f.most_before])
above = pd.concat(above_rows, ignore_index=True)[["model", "date", "weekday", "holiday", "guests_forecast", "most_before"]]
print(f"{len(above)} forecast nights above the most guests that weekday had through Oct 21, 2026.")
above.fillna({"holiday": ""}).style.format(
    {"date": "{:%a %b %-d}", "guests_forecast": "{:.0f}", "most_before": "{:.0f}"}).hide(axis="index")

## 10. Scoring the frozen forecast

Scores the saved file against whatever sales exist after Oct 21, 2026. Add Kieran's next Sales Summary export to `data/sawtooth/` and re-run; the frozen file isn't touched. It also checks the closure assumptions: days expected closed that had sales, and days expected open that didn't. v3's weekly 80% ranges, the only ranges shown in the owner calendar, are scored on weeks where every forecast day has sales; its nightly ranges were frozen before the section 9e widening, so their coverage is reported as history only. If two exports cover the same date, the newest export's row is used.

In [ ]:
saved = pd.read_csv(DAILY_FILE, parse_dates=["date"])
if DAILY_V2.exists():
    saved = saved.merge(pd.read_csv(DAILY_V2, parse_dates=["date"])[["date", "v2_forecast", "v2_lo80", "v2_hi80"]], on="date")
actual = daily.set_index("date").net_sales
saved["actual"] = saved.date.map(actual)
reported = saved[saved.date.isin(actual.index) | (saved.date <= daily.date.max())]
if reported.empty:
    print(f"No sales after {DATA_THROUGH:%b %d, %Y} in data/sawtooth/ yet, so nothing to score.")
else:
    s = reported[reported.expected_open & reported.actual.notna()]
    cols = ["hybrid_forecast", "hybrid_holiday_blend", "linear_forecast", "weekday_median_forecast"]
    cols += ["v2_forecast"] if "v2_forecast" in s else []
    display(pd.Series({c: (s.actual - s[c]).abs().sum() / s.actual.sum() for c in cols},
                      name=f"daily WAPE, {len(s)} open days").to_frame().style.format("{:.1%}"))
    print(f"80% range coverage, v1: {s.actual.between(s.hybrid_lo80, s.hybrid_hi80).mean():.0%} of {len(s)} days")
    if "v2_forecast" in s:
        print(f"80% range coverage, v2: {s.actual.between(s.v2_lo80, s.v2_hi80).mean():.0%} of {len(s)} days")
    wk = s.groupby(s.date.dt.to_period("W-SUN"))[["actual"] + cols].sum()
    display(pd.Series({c: (wk.actual - wk[c]).abs().sum() / wk.actual.sum() for c in cols},
                      name=f"weekly WAPE, {len(wk)} weeks").to_frame().style.format("{:.1%}"))
    if DAILY_V3.exists():
        g = pd.read_csv(DAILY_V3, parse_dates=["date"])
        g["actual"] = g.date.map(daily.set_index("date").guests)
        gs = g[g.expected_open & g.actual.notna()]
        gwk = gs.groupby(gs.date.dt.to_period("W-SUN"))[["actual", "guests_forecast"]].sum()
        print(f"v3 guests: daily WAPE {(gs.actual - gs.guests_forecast).abs().sum() / gs.actual.sum():.1%}, "
              f"weekly WAPE {(gwk.actual - gwk.guests_forecast).abs().sum() / gwk.actual.sum():.1%}, "
              f"frozen nightly 80% range coverage {gs.actual.between(gs.guests_lo80, gs.guests_hi80).mean():.0%} "
              f"(set before the 9e widening, not shown in the calendar)")
        gw = gs[~gs.date.isin(buyout_dates)]
        print(f"v3 guests, walk-in nights only (section 9d rule): daily WAPE {(gw.actual - gw.guests_forecast).abs().sum() / gw.actual.sum():.1%}, "
              f"frozen nightly 80% range coverage {gw.actual.between(gw.guests_lo80, gw.guests_hi80).mean():.0%}")
        if WEEKLY_V3.exists():
            # Weekly ranges sum the expected-open days (buyouts included), so score only weeks where each of those days has an actual.
            og = g[g.expected_open].groupby(g.date.dt.to_period("W-SUN").astype(str)).actual
            w3 = pd.read_csv(WEEKLY_V3).set_index("week").assign(actual=og.sum())[og.apply(lambda a: a.notna().all())]
            print(f"v3 guests, weekly 80% range (the range the calendar shows): {int(w3.actual.between(w3.guests_lo80, w3.guests_hi80).sum())} "
                  f"of {len(w3)} weeks inside")
    if DAILY_V5.exists():
        g5 = pd.read_csv(DAILY_V5, parse_dates=["date"])
        g5["actual"] = g5.date.map(daily.set_index("date").guests)
        g5s = g5[g5.expected_open & g5.actual.notna() & ~g5.date.isin(buyout_dates)]
        if len(g5s):
            w5 = g5s.groupby(g5s.date.dt.to_period("W-SUN"))[["actual", "guests_forecast"]].sum()
            print(f"v5 guests, walk-in nights: daily WAPE {(g5s.actual - g5s.guests_forecast).abs().sum() / g5s.actual.sum():.1%}, "
                  f"weekly WAPE {(w5.actual - w5.guests_forecast).abs().sum() / w5.actual.sum():.1%}")
    print("Expected closed but had sales:", reported[~reported.expected_open & reported.actual.notna()].date.dt.date.tolist())
    print("Expected open but no sales:", reported[reported.expected_open & reported.actual.isna()].date.dt.date.tolist())

## 11. Log-scale baseline (for the next forecast run)

Every model above fits its baseline (weekday, term, 2026 level) to raw guests or sales, so each of those effects is a fixed amount: the 2026 drop takes the same number of guests off a Wednesday as off a Saturday. Everything on top of the baseline (events, holidays, the weekly ranges) is already a percentage, through log ratios. This section fits the baseline to log(guests) instead, so every effect is a percentage, and compares the two on the rolling backtest (refits every 3 months, scored Feb 2 to Oct 18, 2026, as in section 8c).

Converting back with `exp` needs no bias adjustment: the model fits medians, and the median of log(y), exponentiated, is the median of y (Hyndman and Athanasopoulos, *Forecasting: Principles and Practice*, section 5.6). The frozen forecasts in `forecasts/` don't change; this picks the baseline for the next forecast run. It tests v3's design (one 2026 level), not v5's.

**Decision rule.** Written in the second cell below and committed before anyone sees the comparison, as with v5 in 9f. Until the rule exists, the comparison cell computes the scores but doesn't show them. These are the same Feb-Oct 2026 weeks that informed earlier choices, so the check isn't pristine.

In [ ]:
def hybrid_fit_log(train, frame=None, target="guests", base_cols=None, event_cols=None):
    # hybrid_fit (section 8) with the linear baselines fit to log(target), so weekday, term and the 2026 level act as
    # percentages. The trees learn the same thing as before: how events move a day against the median baseline.
    base_cols = BASE if base_cols is None else base_cols
    event_cols = EVENT_INPUTS + ["week_of_term", "weekday"] if event_cols is None else event_cols
    logged = train.assign(log_target=np.log(train[target].clip(lower=FLOOR[target])))
    base = {q: fit_quantile(base_cols, logged, q, frame=frame, target="log_target") for q in (0.1, 0.5, 0.9)}
    model, X = base[0.5]
    resid = logged.log_target - model.predict(X.loc[train.index])
    trees, X_trees = xgb_fit(event_cols, train, resid, frame=frame)
    return base, trees, X_trees


def hybrid_predict_log(fit, rows):
    base, trees, X_trees = fit
    log_multiplier = trees.predict(X_trees.loc[rows.index])
    return {q: pd.Series(np.exp(m.predict(X.loc[rows.index]) + log_multiplier), index=rows.index)
            for q, (m, X) in base.items()}

In [ ]:
def adopt_log_baseline(raw, log):
    """Whether the next forecast run switches to the log baseline. Written before the comparison below is seen.

    raw and log hold guest scores from the rolling backtest, keyed by days ahead (7 and 28). Each is a dict with
    "daily WAPE", "weekly WAPE", "typical week", "weeks >20% off" and "worst week", as fractions (0.113 is 11.3%).
    Return True to switch to the log baseline, False to keep the raw one.
    """
    # Set on 2026-11-09, before any log-baseline score was shown. Switch only if, at both 7 and 28 days ahead, the log
    # baseline (1) cuts weekly WAPE by at least half a point (section 2d treated a 0.7-point gain as possibly luck),
    # (2) has no more weeks over 20% off, and (3) doesn't make daily WAPE worse by more than half a point, since owners
    # see nightly counts too. Anything else, ties included, keeps the raw baseline.
    margin = 0.005
    return all(log[h]["weekly WAPE"] <= raw[h]["weekly WAPE"] - margin
               and log[h]["weeks >20% off"] <= raw[h]["weeks >20% off"]
               and log[h]["daily WAPE"] <= raw[h]["daily WAPE"] + margin for h in (7, 28))

In [ ]:
raw_runs = rolling_scores("guests")
log_runs = rolling_scores("guests", fit_fn=hybrid_fit_log, predict_fn=hybrid_predict_log)
log_table = pd.DataFrame([{"baseline": label, "days ahead": h, **runs[h][0]}
                          for label, runs in (("raw (sections 8-10)", raw_runs), ("log", log_runs)) for h in (7, 28)])
try:
    use_log = adopt_log_baseline({h: raw_runs[h][0] for h in (7, 28)}, {h: log_runs[h][0] for h in (7, 28)})
except NotImplementedError:
    print("The decision rule above isn't written yet, so the comparison stays hidden. Write it, commit it, then re-run.")
else:
    display(log_table.style.format(SCORE_COLS).hide(axis="index"))
    print("Next forecast run's baseline:", "log" if use_log else "raw (unchanged)")

**Takeaway.** The log baseline doesn't pass the rule, so the next forecast run keeps the raw baseline. Weekly WAPE is slightly worse at both horizons (11.5% vs 11.3% a week ahead, 12.3% vs 12.0% four weeks ahead), and daily WAPE is about a point worse (27.9% vs 27.1%, 28.5% vs 27.6%). Its only gain is four weeks ahead, where 14% of weeks were more than 20% off against 16%. Every gap is under a point, so the two are effectively tied, and a tie keeps what's there. The rule was written on 2026-11-09 before these scores were shown, though not committed separately first.